Script 1:

Run Scripts 1 and 2 now to build your initial models based on the 2022-23 and 2023-24 data, and change the years to 2022-24 and 2023-25 data and so on.


In [ ]:
# ==============================================================================
# SCRIPT 1: The Offline Data Processing Engine (FINAL)
# ==============================================================================
# This script loads pre-downloaded box score data, engineers all features
# (player rolling avgs, team rolling DvP), and saves a final, model-ready CSV.
# It is fast, robust, and 100% offline.

# ==============================================================================
# Step 0: Install necessary packages
# ==============================================================================
!pip install pandas unidecode tqdm

# ==============================================================================
# Step 1: Import all necessary libraries
# ==============================================================================
import pandas as pd
import numpy as np
import os
import re
from google.colab import drive
from unidecode import unidecode
from tqdm import tqdm

# ==============================================================================
# Part 2: Helper Functions
# ==============================================================================
def normalize_player_name(name):
    """Cleans and standardizes a player's name for reliable matching."""
    if not isinstance(name, str): return None
    name = unidecode(name.lower()); name = re.sub(r'\s+(jr|sr|ii|iii|iv)\.?$', '', name); name = re.sub(r'[^\w\s]', '', name); name = re.sub(r'\s+', ' ', name).strip()
    return name

def assign_position(pos):
    """Robustly maps the position string to one of the 5 standard positions."""
    if pd.isna(pos): return 'Unknown'
    pos = pos.upper()
    if 'GUARD' in pos or pos == 'G': return 'SG'
    if 'FORWARD' in pos or pos == 'F': return 'PF'
    if 'CENTER' in pos or pos == 'C': return 'C'
    if pos in ['PG', 'SG', 'SF', 'PF', 'C']: return pos
    return 'Unknown'

# ==============================================================================
# Part 3: The Main Data Processing Engine
# ==============================================================================
def process_downloaded_data():
    """
    Main function to load downloaded box scores, engineer all features,
    and save the final training dataset.
    """
    try:
        print("Mounting Google Drive..."); drive.mount('/content/drive', force_remount=True)
        DATA_DIR = "/content/drive/MyDrive/NBA/Historical Data/"
        BOXSCORE_DIR = os.path.join(DATA_DIR, "NocturneBear_Boxscores")
        os.makedirs(DATA_DIR, exist_ok=True)
    except Exception as e:
        print(f"Could not mount Google Drive. Aborting. Error: {e}"); return

    print("Loading and combining downloaded box score files...")
    file_paths = [
        os.path.join(BOXSCORE_DIR, "regular_season_box_scores_2010_2024_part_1.csv"),
        os.path.join(BOXSCORE_DIR, "regular_season_box_scores_2010_2024_part_2.csv"),
        os.path.join(BOXSCORE_DIR, "regular_season_box_scores_2010_2024_part_3.csv")
    ]
    all_dfs = [pd.read_csv(path, low_memory=False) for path in file_paths if os.path.exists(path)]
    if not all_dfs: print("CRITICAL ERROR: No data files found. Aborting."); return
    df = pd.concat(all_dfs, ignore_index=True)
    print(f"Successfully loaded {len(df)} total rows.")

    print("Cleaning and preparing data...")
    df.rename(columns={'season_year': 'SEASON_YEAR', 'game_date': 'GAME_DATE', 'gameId': 'GAME_ID', 'personId': 'PLAYER_ID', 'personName': 'PLAYER_NAME', 'teamTricode': 'TEAM_ABBREVIATION', 'points': 'PTS', 'reboundsTotal': 'REB', 'assists': 'AST', 'minutes': 'MIN', 'fieldGoalsAttempted': 'FGA', 'freeThrowsAttempted': 'FTA', 'threePointersAttempted': 'FG3A', 'steals': 'STL', 'blocks': 'BLK'}, inplace=True)
    df = df[df['MIN'].notna() & (df['MIN'] != '0:00') & (df['MIN'] != '0') & (df['comment'].isna())].copy()
    df = df[df['SEASON_YEAR'].isin(['2022-23', '2023-24'])].copy()
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    df.sort_values(by=['PLAYER_ID', 'GAME_DATE'], inplace=True)
    for col in ['PTS', 'REB', 'AST', 'MIN', 'FGA', 'FTA', 'FG3A', 'STL', 'BLK']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    print("Assigning game-by-game positions...")
    df['Game_Position'] = df['position'].apply(assign_position)

    print("Engineering player rolling average features...")
    stats_to_roll = ['PTS', 'REB', 'AST', 'MIN', 'FGA', 'FTA', 'FG3A']
    windows = [3, 5, 10]
    grouped = df.groupby('PLAYER_ID')
    for stat in tqdm(stats_to_roll, desc="Calculating player averages"):
        for w in windows:
            df[f'ROLL_AVG_{stat}_{w}'] = grouped[stat].transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
    df['DAYS_OF_REST'] = grouped['GAME_DATE'].transform(lambda x: x.diff().dt.days - 1).fillna(7)

    print("Engineering team Defense vs. Position (DvP) features...")
    opponent_df = df[['GAME_ID', 'TEAM_ABBREVIATION']].drop_duplicates()
    df = pd.merge(df, opponent_df.rename(columns={'TEAM_ABBREVIATION': 'OPPONENT_ABBR'}), on='GAME_ID')
    df = df[df['TEAM_ABBREVIATION'] != df['OPPONENT_ABBR']].copy()
    team_allowed_stats = df.groupby(['GAME_ID', 'GAME_DATE', 'OPPONENT_ABBR', 'Game_Position'])[['PTS', 'REB', 'AST']].sum().reset_index()
    team_allowed_stats.rename(columns={'OPPONENT_ABBR': 'Team'}, inplace=True)
    team_allowed_stats.sort_values(by='GAME_DATE', inplace=True)
    final_dvp_features = []
    for pos in ['PG', 'SG', 'SF', 'PF', 'C']:
        pos_df = team_allowed_stats[team_allowed_stats['Game_Position'] == pos].copy()
        grouped = pos_df.groupby('Team')
        for stat in ['PTS', 'REB', 'AST']:
            pos_df[f'OPP_DvP_{stat}_vs_{pos}'] = grouped[stat].transform(lambda x: x.shift(1).rolling(window=30, min_periods=5).mean())
        final_dvp_features.append(pos_df)
    dvp_df = pd.concat(final_dvp_features)
    dvp_cols_to_merge = ['GAME_ID', 'Team', 'Game_Position'] + [col for col in dvp_df if col.startswith('OPP_DvP')]
    df = pd.merge(df, dvp_df[dvp_cols_to_merge], left_on=['GAME_ID', 'OPPONENT_ABBR', 'Game_Position'], right_on=['GAME_ID', 'Team', 'Game_Position'], how='left')
    dvp_cols = [col for col in df.columns if 'OPP_DvP' in col]
    df.sort_values(by=['Team', 'GAME_DATE'], inplace=True)
    df[dvp_cols] = df.groupby('Team')[dvp_cols].transform(lambda x: x.ffill().bfill())
    for stat in ['PTS', 'REB', 'AST']:
        df[f'OPP_DvP_{stat}_vs_POS'] = np.nan
        for pos in ['PG', 'SG', 'SF', 'PF', 'C']:
             df.loc[df['Game_Position'] == pos, f'OPP_DvP_{stat}_vs_POS'] = df[f'OPP_DvP_{stat}_vs_{pos}']

    print("Defining target variables...")
    df['TARGET_PTS'] = df['PTS']; df['TARGET_REB'] = df['REB']; df['TARGET_AST'] = df['AST']
    df['TARGET_P+R'] = df['PTS'] + df['REB']; df['TARGET_P+A'] = df['PTS'] + df['AST']; df['TARGET_R+A'] = df['REB'] + df['AST']
    df['TARGET_PRA'] = df['PTS'] + df['REB'] + df['AST']
    df['DD_COUNT'] = df[['PTS', 'REB', 'AST', 'STL', 'BLK']].ge(10).sum(axis=1)
    df['TARGET_DD'] = (df['DD_COUNT'] >= 2).astype(int); df['TARGET_TD'] = (df['DD_COUNT'] >= 3).astype(int)

    print("Finalizing columns and saving the complete training file...")
    feature_columns = [
        'ROLL_AVG_PTS_3', 'ROLL_AVG_REB_3', 'ROLL_AVG_AST_3', 'ROLL_AVG_PTS_5',
        'ROLL_AVG_REB_5', 'ROLL_AVG_AST_5', 'ROLL_AVG_PTS_10', 'ROLL_AVG_REB_10',
        'ROLL_AVG_AST_10', 'ROLL_AVG_MIN_10', 'ROLL_AVG_FGA_10', 'ROLL_AVG_FTA_10',
        'ROLL_AVG_FG3A_10', 'DAYS_OF_REST',
        'OPP_DvP_PTS_vs_POS', 'OPP_DvP_REB_vs_POS', 'OPP_DvP_AST_vs_POS'
    ]
    target_columns = [
        'TARGET_PTS', 'TARGET_REB', 'TARGET_AST', 'TARGET_P+R', 'TARGET_P+A', 'TARGET_R+A',
        'TARGET_PRA', 'TARGET_DD', 'TARGET_TD'
    ]

    final_df = df[feature_columns + target_columns].copy()
    final_df.dropna(subset=target_columns, inplace=True)

    output_path = os.path.join(DATA_DIR, "nba_training_data_complete.csv")
    final_df.to_csv(output_path, index=False)

    print(f"\n✅ Processing complete! {len(final_df)} model-ready rows saved to:")
    print(output_path)
    display(final_df.head())

if __name__ == "__main__":
    process_downloaded_data()

Mounting Google Drive...
Mounted at /content/drive
Loading and combining downloaded box score files...
Successfully loaded 424478 total rows.
Cleaning and preparing data...
Assigning game-by-game positions...
Engineering player rolling average features...


Calculating player averages: 100%|██████████| 7/7 [00:05<00:00,  1.23it/s]


Engineering team Defense vs. Position (DvP) features...
Defining target variables...
Finalizing columns and saving the complete training file...

✅ Processing complete! 52285 model-ready rows saved to:
/content/drive/MyDrive/NBA/Historical Data/nba_training_data_complete.csv


,ROLL_AVG_PTS_3,ROLL_AVG_REB_3,ROLL_AVG_AST_3,ROLL_AVG_PTS_5,ROLL_AVG_REB_5,ROLL_AVG_AST_5,ROLL_AVG_PTS_10,ROLL_AVG_REB_10,ROLL_AVG_AST_10,ROLL_AVG_MIN_10,ROLL_AVG_FGA_10,ROLL_AVG_FTA_10,ROLL_AVG_FG3A_10,DAYS_OF_REST,OPP_DvP_PTS_vs_POS,OPP_DvP_REB_vs_POS,OPP_DvP_AST_vs_POS,TARGET_PTS,TARGET_REB,TARGET_AST,TARGET_P+R,TARGET_P+A,TARGET_R+A,TARGET_PRA,TARGET_DD,TARGET_TD
1507,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,31.8,9.4,3.8,11,1,4,12,15,5,16,0,0
23103,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,10.6,8.4,3.2,7,9,7,16,14,16,23,0,0
30272,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,33.6,10.4,7.8,21,7,7,28,28,14,35,0,0
37324,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,33.6,10.4,7.8,16,5,3,21,19,8,24,0,0
45422,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,31.8,9.4,3.8,17,7,1,24,18,8,25,0,0


Script 2: V2

Model training

In [ ]:
# ==============================================================================
# SCRIPT 2: The Model Training Engine (FINAL & SIMPLE)
# ==============================================================================
# Run this script exactly as is. Its purpose is to create new model files
# that match the 17-feature list used in your prediction script.

# ==============================================================================
# Step 0: Install necessary packages
# ==============================================================================
!pip install scikit-learn==1.3.0 pandas joblib numpy==1.26.4

# ==============================================================================
# Step 1: Import all necessary libraries
# ==============================================================================
import pandas as pd
import joblib
import json
import os
from google.colab import drive
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_absolute_error, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# ==============================================================================
# Part 2: The Model Training Factory
# ==============================================================================
def train_all_models():
    """
    Loads the pre-processed data from Script 1, trains all models,
    and saves the final model artifacts.
    """
    try:
        print("Mounting Google Drive..."); drive.mount('/content/drive', force_remount=True)
        DATA_DIR = "/content/drive/MyDrive/NBA/Historical Data/"
        MODEL_DIR = "/content/drive/MyDrive/NBA/Training Model"
        os.makedirs(MODEL_DIR, exist_ok=True)
    except Exception as e:
        print(f"Could not mount Google Drive. Aborting. Error: {e}"); return

    input_path = os.path.join(DATA_DIR, "nba_training_data_complete.csv")
    try:
        print(f"Loading training data from: {input_path}")
        df = pd.read_csv(input_path)
    except FileNotFoundError:
        print("CRITICAL ERROR: Training data not found. Please run Script 1 first."); return

    print(f"Loaded {len(df)} rows of model-ready data.")

    features = [
        'ROLL_AVG_PTS_3', 'ROLL_AVG_REB_3', 'ROLL_AVG_AST_3', 'ROLL_AVG_PTS_5',
        'ROLL_AVG_REB_5', 'ROLL_AVG_AST_5', 'ROLL_AVG_PTS_10', 'ROLL_AVG_REB_10',
        'ROLL_AVG_AST_10', 'ROLL_AVG_MIN_10', 'ROLL_AVG_FGA_10', 'ROLL_AVG_FTA_10',
        'ROLL_AVG_FG3A_10', 'DAYS_OF_REST',
        'OPP_DvP_PTS_vs_POS', 'OPP_DvP_REB_vs_POS', 'OPP_DvP_AST_vs_POS'
    ]
    X = df[features]

    regression_props = ['PTS', 'REB', 'AST', 'P+R', 'P+A', 'R+A', 'PRA']
    mae_values = {}
    for prop in regression_props:
        target_column = f'TARGET_{prop}'
        print(f"\n--- Training REGRESSION Model for: {prop} ---")
        y = df[target_column]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('ridge', Ridge())
        ])

        param_grid = {'ridge__alpha': [100.0, 200.0, 500.0, 1000.0]}
        grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
        mae = mean_absolute_error(y_test, best_model.predict(X_test))
        mae_values[prop] = mae
        print(f"  - Final Model MAE for {prop}: {mae:.2f}")
        output_filename = f"nba_{prop.lower()}_ridge_model.pkl"
        joblib.dump(best_model, os.path.join(MODEL_DIR, output_filename))
        print(f"✅ Saved/Overwritten {prop} model.")

    with open(os.path.join(MODEL_DIR, "mae_values.json"), 'w') as f:
        json.dump(mae_values, f, indent=4)
    print(f"\n✅ Saved/Overwritten model MAE values.")

    classification_props = ['DD', 'TD']
    for prop in classification_props:
        target_column = f'TARGET_{prop}'
        print(f"\n--- Training CLASSIFICATION Model for: {prop} ---")
        y = df[target_column]
        if y.nunique() < 2: continue
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

        pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('logreg', LogisticRegression(class_weight='balanced', max_iter=1000))
        ])

        pipeline.fit(X_train, y_train)
        auc = roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1])
        print(f"  - Final Model AUC for {prop}: {auc:.2%}")
        output_filename = f"nba_{prop.lower()}_logreg_model.pkl"
        joblib.dump(pipeline, os.path.join(MODEL_DIR, output_filename))
        print(f"✅ Saved/Overwritten {prop} model.")

# ==============================================================================
# Part 3: Run the script
# ==============================================================================
if __name__ == "__main__":
    train_all_models()

Mounting Google Drive...
Mounted at /content/drive
Loading training data from: /content/drive/MyDrive/NBA/Historical Data/nba_training_data_complete.csv
Loaded 52285 rows of model-ready data.

--- Training REGRESSION Model for: PTS ---


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['ROLL_AVG_MIN_10']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['ROLL_AVG_MIN_10']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


  - Final Model MAE for PTS: 4.66
✅ Saved/Overwritten PTS model.

--- Training REGRESSION Model for: REB ---


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['ROLL_AVG_MIN_10']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['ROLL_AVG_MIN_10']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


  - Final Model MAE for REB: 1.93
✅ Saved/Overwritten REB model.

--- Training REGRESSION Model for: AST ---


...

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['ROLL_AVG_MIN_10']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['ROLL_AVG_MIN_10']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['ROLL_AVG_MIN_10']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


  - Final Model MAE for PRA: 6.20
✅ Saved/Overwritten PRA model.

✅ Saved/Overwritten model MAE values.

--- Training CLASSIFICATION Model for: DD ---


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['ROLL_AVG_MIN_10']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['ROLL_AVG_MIN_10']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


  - Final Model AUC for DD: 87.54%
✅ Saved/Overwritten DD model.

--- Training CLASSIFICATION Model for: TD ---
  - Final Model AUC for TD: 95.08%
✅ Saved/Overwritten TD model.


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:555: UserWarning: Skipping features without any observed values: ['ROLL_AVG_MIN_10']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


Script 3: V2 working but jumbled up team names with wrong players (bad)

In [ ]:
# ==============================================================================
# SCRIPT 3: The Definitive Prediction & Multi-Sheet Output Engine (FINALIZED)
# ==============================================================================

# ==============================================================================
# Step 0: Install all required packages
# ==============================================================================
!pip install selenium xlsxwriter nba_api scikit-learn==1.3.0 pandas joblib numpy==1.26.4 pytz unidecode requests lxml tqdm undetected-chromedriver

# ==============================================================================
# Step 1: Import all necessary libraries
# ==============================================================================
import pandas as pd
import numpy as np
import requests
import re
import time
import os
import joblib
import pytz
import glob
import json
from datetime import datetime, timezone, timedelta
from unidecode import unidecode
from io import StringIO
from google.colab import drive
import undetected_chromedriver as uc
from selenium.webdriver.chrome.options import Options
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, playergamelogs, leaguedashplayerstats, drafthistory, teamestimatedmetrics
from IPython.display import display
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from xlsxwriter.utility import xl_col_to_name
from tqdm import tqdm
import signal
from functools import wraps


# ==============================================================================
# --- CONFIGURATION ---
# ==============================================================================
API_TIMEOUT = 120
API_RETRY_ATTEMPTS = 3
API_RETRY_DELAY = 3
INTER_PLAYER_DELAY = 0.6
WEB_SCRAPE_DELAY = 5
WEB_SCRAPE_RETRY_DELAY = 20
MASTER_SCRAPE_ATTEMPTS = 3
MASTER_SCRAPE_TIMEOUT = 600

ODDS_API_KEYS = [
    "9b96859c5e0781bb4c5809215a5ed94b", "f4cfcea7cc6120a97559f69d6049596e", "7b9405e2febc561e554b7169a42cefd1",
    "146dff42ae19a83208b059d4855095c9", "f8474479351ccc685ccae9f4716c23c5", "39756c093848b6968c899a4e1ded85f9",
    "9705a1f6b609305047eb8ec9243cc54a", "4217574ccdd90732483d98f24772f66b", "66953d96fcb2ec72a47e7f39901258e7",
    "94304f58fb95fd1d45159b9c8d292724", "e384411e342cac5c9408804c622703f1", "dab5c3da80b0415f64d4d94890974cdf",
    "9705a1f6b609305047eb8ec9243cc54a", "3aae94b51cc3dcb8e0623bf2633a695a", "a4e4d3eef7d360055b96a1351560817f",
    "bd6c88c66ff4c7c057aafaffcdd1b8d", "1bfcac154fb2b490bf55a80a18275db1", "7b64d450f91079f60211cb33033594b4",
    "ec9dc1e1aa0543c1b3fa3d2576fabffd", "6bbfe433274aef6fee5aa5543ad69055", "1f5e9cdc190854c605b533917b2c9b82",
    "858b7234b5f77661345f205b1c5ec121"
]
CONFIG = { "GAMES_PLAYED_THRESHOLD_ACCLIMATED": 15, "GAMES_PLAYED_THRESHOLD_LOW_VOLUME": 20 }

# ==============================================================================
# Part 2: Helper Functions and Data Fetching
# ==============================================================================
class TimeoutError(Exception): pass
def run_with_timeout(seconds):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            def handler(signum, frame): raise TimeoutError(f"Function '{func.__name__}' timed out")
            signal.signal(signal.SIGALRM, handler)
            signal.alarm(seconds)
            try: result = func(*args, **kwargs)
            finally: signal.alarm(0)
            return result
        return wrapper
    return decorator

def normalize_player_name(name):
    if not isinstance(name, str): return None
    name = unidecode(name.lower()); name = re.sub(r'\s+(jr|sr|ii|iii|iv)\.?$', '', name); name = re.sub(r'[^\w\s]', '', name); name = re.sub(r'\s+', ' ', name).strip()
    return name

def get_implied_prob_from_decimal(odd):
    if odd is None or pd.isna(odd) or float(odd) < 1: return None
    return 1 / float(odd)

def decimal_to_american(decimal_odds):
    if pd.isna(decimal_odds) or decimal_odds is None: return None
    if decimal_odds >= 2.00: return f"+{int((decimal_odds - 1) * 100)}"
    else: return f"-{int(100 / (decimal_odds - 1))}"

def get_current_nba_season():
    now = datetime.now()
    if now.month >= 8: start_year, end_year = now.year, now.year + 1
    else: end_year, start_year = now.year, now.year - 1
    return f"{start_year}-{str(end_year)[-2:]}"

def fetch_injury_data():
    print("--- Fetching Live Injury Data from ESPN ---")
    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()

        injury_status_dict = {}
        for team in data.get('injuries', []):
            for injury in team.get('injuries', []):
                player_name = injury.get('athlete', {}).get('displayName')
                status = injury.get('type', {}).get('description')
                if player_name and status:
                    normalized_name = normalize_player_name(player_name)
                    injury_status_dict[normalized_name] = status.title()

        print(f"Successfully fetched injury status for {len(injury_status_dict)} players.")
        return injury_status_dict
    except requests.exceptions.RequestException as e:
        print(f"Warning: Could not fetch injury data from ESPN. Error: {e}")
        return {}

def get_draft_class_ids(season_year):
    print(f"--- Fetching {season_year} Draft Class ---")
    try:
        draft_history_data = drafthistory.DraftHistory(season_year_nullable=season_year)
        df_draft = draft_history_data.get_data_frames()[0]
        return set(df_draft['PERSON_ID'].unique())
    except Exception as e:
        print(f"Could not fetch draft class for {season_year}: {e}"); return set()

def identify_team_changes(master_df):
    print("--- Identifying Players on New Teams ---")
    changed_teams_df = master_df[(master_df['TEAM_ABBREVIATION'] != master_df['CURRENT_TEAM']) & (master_df['CURRENT_TEAM'].notna()) & (master_df['TEAM_ABBREVIATION'].notna())]
    return set(changed_teams_df['PLAYER_ID'].unique())

def get_player_experience(row, rookie_ids, traded_player_ids):
    player_id, games_played_last_season, games_played_this_season = row['PLAYER_ID'], row.get('GP', 99), row.get('GP_current', 0)
    is_rookie = player_id in rookie_ids
    is_traded = player_id in traded_player_ids
    is_acclimated = games_played_this_season >= CONFIG["GAMES_PLAYED_THRESHOLD_ACCLIMATED"]
    if is_rookie: return "Traded Rookie" if is_traded and not is_acclimated else "Rookie"
    else:
        if is_traded and not is_acclimated: return "Traded Veteran"
        if games_played_last_season < CONFIG["GAMES_PLAYED_THRESHOLD_LOW_VOLUME"]: return "Low Volume"
        return "Veteran"

@run_with_timeout(MASTER_SCRAPE_TIMEOUT)
def scrape_bref_positions_by_team(season_end_year, teams_to_scrape):
    print(f"--- Scraping B-Ref rosters for {len(teams_to_scrape)} teams for season {season_end_year} ---")
    all_players_df_list = []
    chrome_options = uc.ChromeOptions(); chrome_options.add_argument('--headless'); chrome_options.add_argument('--no-sandbox'); chrome_options.add_argument('--disable-dev-shm-usage'); chrome_options.add_argument('--disable-gpu')
    driver = uc.Chrome(options=chrome_options)
    try:
        for team_abbr_bref in tqdm(teams_to_scrape, desc="Scraping B-Ref Positions"):
            for attempt in range(API_RETRY_ATTEMPTS):
                try:
                    url = f"https://www.basketball-reference.com/teams/{team_abbr_bref}/{season_end_year}.html"
                    driver.get(url)
                    WebDriverWait(driver, 30).until(EC.presence_of_element_located((By.ID, "roster")))
                    tables = pd.read_html(StringIO(driver.page_source), attrs={'id': 'roster'})
                    if tables:
                        team_df = tables[0].copy()
                        team_df['CURRENT_TEAM_BRef'] = team_abbr_bref
                        team_df.rename(columns={'Player': 'BRef_Name', 'Pos': 'DetailedPos'}, inplace=True)
                        all_players_df_list.append(team_df)
                    time.sleep(WEB_SCRAPE_DELAY)
                    break
                except Exception as e:
                    print(f"  - Attempt {attempt+1} failed for {team_abbr_bref}: {e}")
                    if attempt < API_RETRY_ATTEMPTS - 1:
                        time.sleep(WEB_SCRAPE_RETRY_DELAY)
                    else:
                        print(f"  - CRITICAL: All attempts failed for {team_abbr_bref}. Skipping team.")
    finally:
        driver.quit()
    if not all_players_df_list: print("CRITICAL ERROR: Failed to scrape any team data."); return pd.DataFrame()
    return pd.concat(all_players_df_list, ignore_index=True)

def update_and_get_live_dvp(season, data_dir, bref_positions_df, nba_player_stats_df):
    print("\n--- Updating and Calculating Live DvP Data ---")
    gamelogs_cache_path = os.path.join(data_dir, f"live_gamelogs_{season}.csv")
    existing_gamelogs = pd.DataFrame()
    last_date_in_cache = None
    if os.path.exists(gamelogs_cache_path):
        try:
            existing_gamelogs = pd.read_csv(gamelogs_cache_path, low_memory=False)
            existing_gamelogs['GAME_DATE'] = pd.to_datetime(existing_gamelogs['GAME_DATE'])
            if not existing_gamelogs.empty:
                last_date_in_cache = existing_gamelogs['GAME_DATE'].max()
                print(f"Loaded {len(existing_gamelogs)} cached gamelogs. Last entry is from: {last_date_in_cache.date()}")
        except Exception as e: print(f"Warning: Could not read gamelogs cache. Error: {e}")

    date_from = (last_date_in_cache + timedelta(days=1)).strftime('%m/%d/%Y') if last_date_in_cache else None

    try:
        print(f"Fetching new game logs from {date_from or 'the beginning of the season'}...")
        new_gamelogs = playergamelogs.PlayerGameLogs(season_nullable=season, date_from_nullable=date_from, timeout=API_TIMEOUT).get_data_frames()[0]
        if not new_gamelogs.empty:
            new_gamelogs['GAME_DATE'] = pd.to_datetime(new_gamelogs['GAME_DATE'])
            all_gamelogs = pd.concat([existing_gamelogs, new_gamelogs], ignore_index=True)
            all_gamelogs.drop_duplicates(subset=['GAME_ID', 'PLAYER_ID'], keep='last', inplace=True)
            all_gamelogs.to_csv(gamelogs_cache_path, index=False)
            print(f"Found {len(new_gamelogs)} new game logs. Cache updated.")
        else:
            print("No new game logs found. DvP data is up to date.")
            all_gamelogs = existing_gamelogs
    except Exception as e:
        print(f"Warning: Failed to fetch new game logs. Using cached data. Error: {e}")
        all_gamelogs = existing_gamelogs

    if all_gamelogs.empty: print("CRITICAL: No game logs available to calculate DvP."); return pd.DataFrame()

    print("Calculating season-to-date DvP stats...")

    gamelogs_with_pos = pd.DataFrame()
    if not bref_positions_df.empty:
        print("Using B-Ref for detailed positions.")
        bref_positions_df['BRef_Name_Normalized'] = bref_positions_df['BRef_Name'].apply(normalize_player_name)
        all_gamelogs['PLAYER_NAME_NORMALIZED'] = all_gamelogs['PLAYER_NAME'].apply(normalize_player_name)
        gamelogs_with_pos = pd.merge(all_gamelogs, bref_positions_df[['BRef_Name_Normalized', 'DetailedPos']], left_on='PLAYER_NAME_NORMALIZED', right_on='BRef_Name_Normalized', how='left')
        gamelogs_with_pos.rename(columns={'DetailedPos': 'Game_Position'}, inplace=True)
        gamelogs_with_pos['Game_Position'] = gamelogs_with_pos['Game_Position'].apply(lambda x: x.split('-')[0] if isinstance(x, str) and x else 'Unknown')
    elif not nba_player_stats_df.empty:
        print("B-Ref scrape failed. Using NBA_API for general positions (G, F, C).")
        gamelogs_with_pos = pd.merge(all_gamelogs, nba_player_stats_df[['PLAYER_ID', 'POSITION']], on='PLAYER_ID', how='left')
        gamelogs_with_pos.rename(columns={'POSITION': 'Game_Position'}, inplace=True)
        gamelogs_with_pos['Game_Position'] = gamelogs_with_pos['Game_Position'].apply(lambda x: x.split('-')[0] if isinstance(x, str) and x else 'Unknown')
    else:
        print("CRITICAL WARNING: No position data available. DvP will be inaccurate.")
        print("CRITICAL WARNING: No position data available. DvP will be inaccurate.")
        print(f"nba_positions_df columns: {nba_positions_df.columns.tolist()}")
        print(f"nba_positions_df rows: {len(nba_positions_df)}")

        return pd.DataFrame()

    opponent_df = gamelogs_with_pos[['GAME_ID', 'TEAM_ABBREVIATION']].drop_duplicates()
    gamelogs_with_pos = pd.merge(gamelogs_with_pos, opponent_df.rename(columns={'TEAM_ABBREVIATION': 'OPPONENT_ABBR'}), on='GAME_ID')
    gamelogs_with_pos = gamelogs_with_pos[gamelogs_with_pos['TEAM_ABBREVIATION'] != gamelogs_with_pos['OPPONENT_ABBR']]
    for stat in ['PTS', 'REB', 'AST']: gamelogs_with_pos[stat] = pd.to_numeric(gamelogs_with_pos[stat], errors='coerce')
    dvp = gamelogs_with_pos.groupby(['OPPONENT_ABBR', 'Game_Position'])[['PTS', 'REB', 'AST']].mean().reset_index()
    dvp_pivot = dvp.pivot_table(index='OPPONENT_ABBR', columns='Game_Position', values=['PTS', 'REB', 'AST'])
    dvp_pivot.columns = [f'OPP_DvP_{stat}_vs_{pos}' for stat, pos in dvp_pivot.columns]
    print("--- DvP Calculation Complete ---")
    return dvp_pivot

feature_cache = {}
def get_live_features_for_player(player_id, stats_season):
    if player_id in feature_cache: return feature_cache[player_id]
    print(f"  - Engineering features for player ID: {player_id}...")
    try:
        logs = playergamelog.PlayerGameLog(player_id=player_id, season=stats_season, timeout=API_TIMEOUT).get_data_frames()[0]
        if logs.empty or len(logs) < 3: return None
        logs['GAME_DATE'] = pd.to_datetime(logs['GAME_DATE']); logs = logs.sort_values('GAME_DATE', ascending=True)
        features = {}
        stats_to_roll = ['PTS', 'REB', 'AST', 'MIN', 'FGA', 'FTA', 'FG3A']
        windows = [3, 5, 10]
        for stat in stats_to_roll:
            for w in windows: features[f'ROLL_AVG_{stat}_{w}'] = logs[stat].tail(w).mean()

        # --- CORRECTED: Backfill missing longer-term averages with shorter-term ones in the correct order ---
        for stat in stats_to_roll:
            # First, ensure 5-game avg is filled if it's missing
            if pd.isna(features.get(f'ROLL_AVG_{stat}_5')):
                features[f'ROLL_AVG_{stat}_5'] = features.get(f'ROLL_AVG_{stat}_3')
            # Now, ensure 10-game avg is filled with the (potentially newly filled) 5-game avg
            if pd.isna(features.get(f'ROLL_AVG_{stat}_10')):
                features[f'ROLL_AVG_{stat}_10'] = features.get(f'ROLL_AVG_{stat}_5')

        features['DAYS_OF_REST'] = min((datetime.now(timezone.utc).replace(tzinfo=None) - logs.iloc[-1]['GAME_DATE']).days, 7)
        feature_cache[player_id] = features
        return features
    except Exception as e:
        print(f"    - Could not fetch game logs for player {player_id}. Error: {e}"); return None

# ==============================================================================
# Part 3: The Main Prediction Engine
# ==============================================================================
def run_daily_projections():
    try:
        drive.mount('/content/drive', force_remount=True)
        DATA_DIR = "/content/drive/MyDrive/NBA/Historical Data/"
        MODEL_DIR = "/content/drive/MyDrive/NBA/Training Model"
        OUTPUT_DIR = "/content/drive/MyDrive/NBA/Predictions"
        os.makedirs(MODEL_DIR, exist_ok=True); os.makedirs(OUTPUT_DIR, exist_ok=True); os.makedirs(DATA_DIR, exist_ok=True)
    except Exception as e: print(f"Could not mount Google Drive. Aborting. Error: {e}"); return

    injury_data = fetch_injury_data()
    print("\n--- Loading Trained Models & Performance Data ---")
    regression_props = ['PTS', 'REB', 'AST', 'P+R', 'P+A', 'R+A', 'PRA']; classification_props = ['DD', 'TD']
    models = {}; mae_values = {}
    try:
        for prop in regression_props: models[prop] = joblib.load(os.path.join(MODEL_DIR, f"nba_{prop.lower()}_ridge_model.pkl"))
        for prop in classification_props: models[prop] = joblib.load(os.path.join(MODEL_DIR, f"nba_{prop.lower()}_logreg_model.pkl"))
        with open(os.path.join(MODEL_DIR, "mae_values.json"), 'r') as f: mae_values = json.load(f)
    except FileNotFoundError as e: print(f"CRITICAL ERROR: Could not load model files. Please run Script 2. Missing: {e}"); return

    CURRENT_SEASON = get_current_nba_season()
    eastern = pytz.timezone('US/Eastern'); now_et = datetime.now(eastern)

    print("\n--- Step 1: Fetching today's game schedule from Odds API ---")
    start_of_day_et = now_et.replace(hour=8, minute=0, second=0, microsecond=0); end_of_day_et = start_of_day_et + timedelta(days=1)
    start_utc_str = start_of_day_et.astimezone(pytz.utc).strftime('%Y-%m-%dT%H:%M:%SZ'); end_utc_str = end_of_day_et.astimezone(pytz.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    working_key, events = None, []
    for i, api_key in enumerate(ODDS_API_KEYS):
        try:
            response = requests.get('https://api.the-odds-api.com/v4/sports/basketball_nba/events', params={'apiKey': api_key, 'commenceTimeFrom': start_utc_str, 'commenceTimeTo': end_utc_str}, timeout=30)
            response.raise_for_status(); events = response.json(); working_key = api_key; break
        except Exception as e: print(f"--- API Key #{i+1} failed. Error: {e} ---")
    if not working_key: print("--- CRITICAL: All API keys failed. ---"); return
    if not events: print("No NBA games scheduled for today."); return
    team_name_map = {team['full_name']: team['abbreviation'] for team in teams.get_teams()}
    games_today = [{'home_team_abbr': team_name_map.get(e['home_team']), 'away_team_abbr': team_name_map.get(e['away_team'])} for e in events if team_name_map.get(e['home_team']) and team_name_map.get(e['away_team'])]
    games_today = [dict(t) for t in {tuple(d.items()) for d in games_today}]

    print("\n--- Step 2: Acquiring Player Rosters & Positions (with self-healing) ---")
    today_str = now_et.strftime('%Y-%m-%d')
    current_year_start = int(CURRENT_SEASON.split('-')[0])
    POSITIONS_SEASON_END_YEAR = current_year_start + 1
    teams_playing_today = {g['home_team_abbr'] for g in games_today} | {g['away_team_abbr'] for g in games_today}
    bref_team_map = {"BKN": "BRK", "PHX": "PHO", "CHA": "CHO"}; teams_to_scrape_bref = [bref_team_map.get(team, team) for team in teams_playing_today]

    bref_df = pd.DataFrame()
    for attempt in range(MASTER_SCRAPE_ATTEMPTS):
        try:
            print(f"Master scraping attempt {attempt + 1}/{MASTER_SCRAPE_ATTEMPTS}...")
            bref_df = scrape_bref_positions_by_team(season_end_year=POSITIONS_SEASON_END_YEAR, teams_to_scrape=teams_to_scrape_bref)
            if not bref_df.empty and set(bref_df['CURRENT_TEAM_BRef'].unique()) == set(teams_to_scrape_bref):
                print("✅ Successfully scraped all required team rosters.")
                break
            else:
                print(f"  - Warning: Scrape attempt {attempt+1} was incomplete. Retrying...")
        except TimeoutError as e:
            print(f"  - CRITICAL: Scrape attempt {attempt+1} timed out. Error: {e}")
        except Exception as e:
            print(f"  - CRITICAL: Scrape attempt {attempt+1} failed with an unexpected error: {e}")
        if attempt < MASTER_SCRAPE_ATTEMPTS - 1:
            print("    -> Waiting before next master attempt...")
            time.sleep(WEB_SCRAPE_RETRY_DELAY)

    if bref_df.empty:
        print("CRITICAL FAILURE: Could not scrape rosters. Will use NBA_API for general positions for DvP.")

    use_previous_season_stats = today_str < f"{current_year_start}-12-01"
    STATS_SEASON_STRING = f"{current_year_start - 1}-{str(current_year_start)[-2:]}" if use_previous_season_stats else CURRENT_SEASON
    nba_df = leaguedashplayerstats.LeagueDashPlayerStats(season=STATS_SEASON_STRING, per_mode_detailed="PerGame").get_data_frames()[0]
    nba_df_current = leaguedashplayerstats.LeagueDashPlayerStats(season=CURRENT_SEASON, per_mode_detailed="PerGame").get_data_frames()[0] if use_previous_season_stats and CURRENT_SEASON != STATS_SEASON_STRING else nba_df

    dvp_df = update_and_get_live_dvp(CURRENT_SEASON, DATA_DIR, bref_df, nba_df)

    print("\n--- Step 3: Fetching player prop odds from Odds API ---")
    MARKETS_TO_KEEP = {'player_points': 'PTS', 'player_rebounds': 'REB', 'player_assists': 'AST', 'player_points_rebounds_assists': 'PRA', 'player_points_rebounds': 'P+R', 'player_points_assists': 'P+A', 'player_rebounds_assists': 'R+A', 'player_double_double': 'DD', 'player_triple_double': 'TD'}
    all_props = [];
    key_usage = {}
    print("Checking API key usage...")
    for key in ODDS_API_KEYS:
        try:
            usage_response = requests.get('https://api.the-odds-api.com/v4/sports', params={'apiKey': key})
            remaining = int(usage_response.headers.get('x-requests-remaining', 0))
            key_usage[key] = remaining
            print(f"  - Key ending in ...{key[-4:]} has {remaining} requests remaining.")
        except Exception as e:
            key_usage[key] = 0
            print(f"  - Could not check usage for key ending in ...{key[-4:]}. Error: {e}")

    current_key_index = 0
    for event in events:
        event_id = event['id']
        odds_fetched = False
        while not odds_fetched and current_key_index < len(ODDS_API_KEYS):
            api_key = ODDS_API_KEYS[current_key_index]
            if key_usage.get(api_key, 0) < len(events):
                print(f"Skipping key ...{api_key[-4:]}, not enough requests remaining.")
                current_key_index += 1
                continue
            params = {'apiKey': api_key, 'regions': 'us', 'markets': ','.join(MARKETS_TO_KEEP.keys()), 'bookmakers': 'draftkings,fanduel'}
            try:
                response = requests.get(f'https://api.the-odds-api.com/v4/sports/basketball_nba/events/{event_id}/odds', params=params, timeout=30)
                response.raise_for_status()
                for bookmaker in response.json().get('bookmakers', []):
                    for market in bookmaker.get('markets', []):
                        prop_type = MARKETS_TO_KEEP.get(market['key'])
                        if prop_type:
                            for outcome in market.get('outcomes', []):
                                all_props.append({ 'Game': f"{team_name_map.get(event['away_team'])} @ {team_name_map.get(event['home_team'])}", 'Game_Time': event['commence_time'], 'Player': outcome['description'], 'Prop': prop_type, 'Line': outcome.get('point'), 'Over_Odds': outcome['price'] if outcome['name'] == 'Over' else None, 'Under_Odds': outcome['price'] if outcome['name'] == 'Under' else None, 'Yes_Odds': outcome['price'] if outcome['name'] == 'Yes' else None, 'No_Odds': outcome['price'] if outcome['name'] == 'No' else None, 'Bookmaker': bookmaker['title'] })
                odds_fetched = True
            except requests.exceptions.RequestException as e:
                print(f"Error fetching odds for event {event_id}. Key #{current_key_index + 1} failed. Error: {e}")
                current_key_index += 1
        if not odds_fetched and current_key_index >= len(ODDS_API_KEYS):
            print("All Odds API keys failed. Aborting odds fetch for remaining games.")
            break

    if not all_props: print("No player props were available. Aborting."); return
    df = pd.DataFrame(all_props); df['match_key'] = df['Player'].apply(normalize_player_name)
    odds_df_ou = df[df['Prop'].isin(regression_props)].groupby(['match_key', 'Player', 'Prop', 'Line', 'Bookmaker', 'Game', 'Game_Time']).agg({'Over_Odds': 'first', 'Under_Odds': 'first'}).reset_index()
    odds_df_yn = df[df['Prop'].isin(classification_props)].groupby(['match_key', 'Player', 'Prop', 'Bookmaker', 'Game', 'Game_Time']).agg({'Yes_Odds': 'first', 'No_Odds': 'first'}).reset_index()
    odds_df_yn['Implied_Prob_Yes'] = odds_df_yn['Yes_Odds'].apply(get_implied_prob_from_decimal)
    odds_df = pd.concat([odds_df_ou, odds_df_yn], ignore_index=True)
    for col in ['Over_Odds', 'Under_Odds', 'Yes_Odds', 'No_Odds']: odds_df[col] = odds_df[col].apply(decimal_to_american)
    odds_df['Game_Time'] = pd.to_datetime(odds_df['Game_Time']).dt.tz_convert('US/Eastern').dt.strftime('%-I:%M %p EST')

    print("\n--- Step 4: Preparing Data for Projections ---")
    nba_df['merge_key'] = nba_df['PLAYER_NAME'].apply(normalize_player_name)

    if not bref_df.empty:
        bref_df['merge_key'] = bref_df['BRef_Name'].apply(normalize_player_name)
        master_df = pd.merge(nba_df, bref_df, on='merge_key', how='inner')
        reverse_bref_map = {v: k for k, v in bref_team_map.items()}
        master_df['CURRENT_TEAM'] = master_df['CURRENT_TEAM_BRef'].apply(lambda x: reverse_bref_map.get(x, x))
    else:
        master_df = nba_df.copy()
        master_df['DetailedPos'] = master_df['POSITION']
        master_df['CURRENT_TEAM'] = master_df['TEAM_ABBREVIATION']


    if use_previous_season_stats: master_df['GP_current'] = master_df['PLAYER_ID'].map(nba_df_current.set_index('PLAYER_ID')['GP']).fillna(0)
    else: master_df['GP_current'] = master_df['GP']
    rookie_ids = get_draft_class_ids(str(current_year_start))
    traded_player_ids = identify_team_changes(master_df)

    players_with_odds = odds_df['match_key'].unique()
    players_to_project_df = master_df[master_df['merge_key'].isin(players_with_odds)]
    print(f"\n--- Step 5: Generating Projections for {len(players_to_project_df)} Players with Active Props ---")

    all_projections = []
    for _, player in players_to_project_df.iterrows():
        time.sleep(INTER_PLAYER_DELAY)
        player_features = get_live_features_for_player(player['PLAYER_ID'], CURRENT_SEASON)
        if not player_features: continue

        opp_team_abbr = next((g['home_team_abbr'] if player['CURRENT_TEAM'] == g['away_team_abbr'] else g['away_team_abbr'] for g in games_today if player['CURRENT_TEAM'] in [g['home_team_abbr'], g['away_team_abbr']]), None)
        if not opp_team_abbr: continue

        player_pos = player['DetailedPos'].split('-')[0]
        for stat in ['PTS', 'REB', 'AST']:
            dvp_feature_name = f'OPP_DvP_{stat}_vs_POS'
            dvp_lookup_col = f'OPP_DvP_{stat}_vs_{player_pos}'
            if not dvp_df.empty and opp_team_abbr in dvp_df.index and dvp_lookup_col in dvp_df.columns:
                player_features[dvp_feature_name] = dvp_df.loc[opp_team_abbr, dvp_lookup_col]
            else:
                player_features[dvp_feature_name] = np.nan

        projections_for_player = {'Player': player['PLAYER_NAME'], 'Team': player['CURRENT_TEAM'], 'Pos': player['DetailedPos'], 'match_key': player['merge_key'], 'Experience': get_player_experience(player, rookie_ids, traded_player_ids)}
        projections_for_player['Injury'] = injury_data.get(player['merge_key'], 'Healthy')

        TRAINING_FEATURES = [
            'ROLL_AVG_PTS_3', 'ROLL_AVG_REB_3', 'ROLL_AVG_AST_3', 'ROLL_AVG_PTS_5',
            'ROLL_AVG_REB_5', 'ROLL_AVG_AST_5', 'ROLL_AVG_PTS_10', 'ROLL_AVG_REB_10',
            'ROLL_AVG_AST_10', 'ROLL_AVG_MIN_10', 'ROLL_AVG_FGA_10', 'ROLL_AVG_FTA_10',
            'ROLL_AVG_FG3A_10', 'DAYS_OF_REST',
            'OPP_DvP_PTS_vs_POS', 'OPP_DvP_REB_vs_POS', 'OPP_DvP_AST_vs_POS'
        ]
        feature_df = pd.DataFrame([player_features])
        feature_df = feature_df[TRAINING_FEATURES]

        for prop in regression_props: projections_for_player[f'Proj_{prop}'] = round(max(0, models[prop].predict(feature_df)[0]), 1)
        for prop in classification_props: projections_for_player[f'Proj_{prop}_%'] = f"{models[prop].predict_proba(feature_df)[0][1]:.1%}"
        all_projections.append(projections_for_player)

    if not all_projections: print("\n--- WARNING: No projections were generated. ---"); return
    projections_df = pd.DataFrame(all_projections)

    print("\n--- Step 6: Calculating Edge and Generating Output Files ---")
    melted_projections = projections_df.melt(id_vars=['match_key', 'Player', 'Team', 'Injury', 'Experience', 'Pos'], value_vars=[c for c in projections_df.columns if c.startswith('Proj')], var_name='Prop_Name', value_name='Projection')
    melted_projections['Prop'] = melted_projections['Prop_Name'].str.replace('Proj_', '').str.replace('_%', '')
    master_long_df = pd.merge(melted_projections, odds_df, on=['match_key', 'Prop'], how='left', suffixes=('_proj', '_odds'))
    master_long_df.dropna(subset=['Game'], inplace=True)
    master_long_df['Player'] = master_long_df['Player_proj'].fillna(master_long_df['Player_odds']); master_long_df.drop(columns=['Player_proj', 'Player_odds'], inplace=True, errors='ignore')

    # --- NEW: Robust Sanity Check to filter out scrambled player data ---
    def get_teams_from_game_str(game_str):
        if not isinstance(game_str, str) or '@' not in game_str: return [], []
        away, home = game_str.split(' @ ')
        return away.strip(), home.strip()

    teams_in_game = master_long_df['Game'].apply(get_teams_from_game_str)
    master_long_df['AwayTeam'] = teams_in_game.apply(lambda x: x[0])
    master_long_df['HomeTeam'] = teams_in_game.apply(lambda x: x[1])

    initial_rows = len(master_long_df)
    master_long_df = master_long_df[master_long_df.apply(lambda row: row['Team'] in [row['AwayTeam'], row['HomeTeam']], axis=1)]
    filtered_rows = initial_rows - len(master_long_df)
    if filtered_rows > 0:
        print(f"  - Sanity Check: Removed {filtered_rows} rows with incorrect player/team assignments due to placeholder API data.")
    master_long_df.drop(columns=['AwayTeam', 'HomeTeam'], inplace=True)

    master_long_df['Projection_Num'] = pd.to_numeric(master_long_df['Projection'].astype(str).str.replace('%', ''), errors='coerce')
    master_long_df.drop(columns=['Projection'], inplace=True, errors='ignore')

    yn_mask = master_long_df['Prop'].isin(classification_props)
    ou_mask = master_long_df['Prop'].isin(regression_props)

    master_long_df['DD & TD\nModel Score'] = np.nan
    master_long_df.loc[yn_mask, 'DD & TD\nModel Score'] = master_long_df.loc[yn_mask, 'Projection_Num']
    master_long_df.loc[yn_mask, 'Projection_Num'] = np.nan

    for prop, mae in mae_values.items():
        mask = master_long_df['Prop'] == prop
        master_long_df.loc[mask, 'Proj Range (Low)'] = (master_long_df.loc[mask, 'Projection_Num'] - mae).round(1).apply(lambda x: max(0, x))
        master_long_df.loc[mask, 'Proj Range (High)'] = (master_long_df.loc[mask, 'Projection_Num'] + mae).round(1)
    master_long_df['Proj Range'] = master_long_df.apply(lambda row: f"{row['Proj Range (Low)']}-{row['Proj Range (High)']}" if pd.notna(row['Proj Range (Low)']) else "", axis=1)

    master_long_df['Edge'] = None; master_long_df['Bet'] = 'Pass'
    master_long_df.loc[ou_mask, 'Edge'] = ((master_long_df.loc[ou_mask, 'Projection_Num'] - master_long_df.loc[ou_mask, 'Line']) / master_long_df.loc[ou_mask, 'Line']) * 100
    master_long_df.loc[ou_mask, 'Bet'] = master_long_df.loc[ou_mask, 'Edge'].apply(lambda x: 'Over' if pd.notna(x) and x > 10 else ('Under' if pd.notna(x) and x < -10 else 'Pass'))
    if 'Implied_Prob_Yes' in master_long_df.columns:
        master_long_df.loc[yn_mask, 'Edge'] = master_long_df.loc[yn_mask, 'DD & TD\nModel Score'] - (master_long_df.loc[yn_mask, 'Implied_Prob_Yes'].fillna(0) * 100)
    else:
        master_long_df.loc[yn_mask, 'Edge'] = None
    master_long_df.loc[yn_mask, 'Bet'] = master_long_df.loc[yn_mask, 'Edge'].apply(lambda x: 'Yes' if pd.notna(x) and x > 5 else ('No' if pd.notna(x) and x < -5 else 'Pass'))
    if 'Implied_Prob_Yes' in master_long_df.columns: master_long_df.drop(columns=['Implied_Prob_Yes'], inplace=True)


    master_long_df['Alt Line'] = 'Pass'
    alt_line_mask = (master_long_df['Bet'] == 'Over') & (master_long_df['Proj Range (Low)'].notna())
    alt_line_values = np.floor(pd.to_numeric(master_long_df.loc[alt_line_mask, 'Proj Range (Low)'], errors='coerce'))
    valid_alt_line_mask = alt_line_values < master_long_df.loc[alt_line_mask, 'Line']
    master_long_df.loc[alt_line_mask & valid_alt_line_mask, 'Alt Line'] = 'Over ' + alt_line_values[valid_alt_line_mask].astype(int).astype(str)

    master_long_df['DD/TD Odds'] = master_long_df['Yes_Odds']
    master_long_df.rename(columns={'Projection_Num': 'Projection', 'Over_Odds': 'Over Odds', 'Under_Odds': 'Under Odds'}, inplace=True)
    master_long_df.drop(columns=['Proj Range (Low)', 'Proj Range (High)', 'Yes_Odds', 'No_Odds'], inplace=True, errors='ignore')

    cols_to_round = ['DD & TD\nModel Score', 'Line', 'Projection', 'Edge']
    for col in cols_to_round:
        if col in master_long_df.columns:
            master_long_df[col] = pd.to_numeric(master_long_df[col], errors='coerce').round(1)

    game_date_str = pd.to_datetime(events[0]['commence_time']).tz_convert('US/Eastern').strftime('%m-%d-%Y')
    output_filename = f"{game_date_str} NBA Props Predictions.xlsx"
    full_output_path = os.path.join(OUTPUT_DIR, output_filename)
    with pd.ExcelWriter(full_output_path, engine='xlsxwriter') as writer:
        workbook  = writer.book
        header_format = workbook.add_format({'bold': True, 'align': 'center', 'valign': 'vcenter', 'fg_color': '#D7E4BC', 'border': 1, 'text_wrap': True})
        center_format = workbook.add_format({'align': 'center', 'valign': 'vcenter', 'border': 1})
        injured_format = workbook.add_format({'bg_color': '#FFEB9C', 'align': 'center', 'valign': 'vcenter', 'border': 1}) # Yellow fill for injured players

        ev_cols = ['Game_Time', 'Game', 'Player', 'Team', 'Experience', 'Prop', 'DD & TD\nModel Score', 'Line', 'Projection', 'Proj Range', 'Alt Line', 'Edge', 'Bet', 'Over Odds', 'Under Odds', 'DD/TD Odds', 'Injury']
        final_cols = ['Player', 'Team', 'Experience', 'Prop', 'DD & TD\nModel Score', 'Line', 'Projection', 'Proj Range', 'Alt Line', 'Edge', 'Bet', 'Over Odds', 'Under Odds', 'DD/TD Odds', 'Bookmaker', 'Injury']

        all_ev_bets_df = master_long_df[master_long_df['Bet'] != 'Pass'].copy()
        if not all_ev_bets_df.empty:
            for bookmaker in ['DraftKings', 'FanDuel']:
                book_df = all_ev_bets_df[all_ev_bets_df['Bookmaker'] == bookmaker]
                if book_df.empty: continue
                sorted_games = sorted(games_today, key=lambda g: (g['home_team_abbr'], g['away_team_abbr']))
                game_dfs = [game_df.sort_values(by='Edge', key=abs, ascending=False) for game_tuple in sorted_games if not (game_df := book_df[book_df['Game'] == f"{game_tuple['away_team_abbr']} @ {game_tuple['home_team_abbr']}"]).empty]
                if game_dfs:
                    final_ev_df = pd.concat(game_dfs, ignore_index=True)
                    final_ev_df = final_ev_df.reindex(columns=ev_cols)
                    sheet_name = f"{bookmaker}_EV_Bets"
                    final_ev_df.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=1)
                    worksheet = writer.sheets[sheet_name]

                    for col_num, value in enumerate(final_ev_df.columns.values):
                        worksheet.write(0, col_num, value, header_format)
                        data_len = final_ev_df[value].astype(str).map(len).max() if not final_ev_df[value].isnull().all() else 0
                        header_len = max([len(s) for s in value.split('\n')]) if '\n' in value else len(value)
                        column_width = max(data_len, header_len, 10) + 2
                        worksheet.set_column(col_num, col_num, column_width, center_format)
                    worksheet.set_row(0, 30)

                    # Add conditional formatting for injured players
                    injury_col_letter = xl_col_to_name(ev_cols.index('Injury'))
                    worksheet.conditional_format(f'A2:{xl_col_to_name(len(ev_cols)-1)}{len(final_ev_df)+1}', {
                        'type': 'formula',
                        'formula': f'${injury_col_letter}2 <> "Healthy"',
                        'format': injured_format
                    })


        for game in sorted(games_today, key=lambda g: (g['home_team_abbr'], g['away_team_abbr'])):
            home_abbr, away_abbr = game['home_team_abbr'], game['away_team_abbr']
            sheet_name = f"{away_abbr}_vs_{home_abbr}_Full_Data"
            game_df = master_long_df[master_long_df['Game'] == f"{away_abbr} @ {home_abbr}"].copy()
            if game_df.empty: continue
            game_long_df = game_df.reindex(columns=final_cols).sort_values(by=['Player', 'Prop'])
            game_long_df.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=1)
            worksheet = writer.sheets[sheet_name]

            for col_num, value in enumerate(game_long_df.columns.values):
                worksheet.write(0, col_num, value, header_format)
                data_len = game_long_df[value].astype(str).map(len).max() if not game_long_df[value].isnull().all() else 0
                header_len = max([len(s) for s in value.split('\n')]) if '\n' in value else len(value)
                column_width = max(data_len, header_len, 10) + 2
                worksheet.set_column(col_num, col_num, column_width, center_format)
            worksheet.set_row(0, 30)

            # Add conditional formatting for injured players
            injury_col_letter = xl_col_to_name(final_cols.index('Injury'))
            worksheet.conditional_format(f'A2:{xl_col_to_name(len(final_cols)-1)}{len(game_long_df)+1}', {
                'type': 'formula',
                'formula': f'${injury_col_letter}2 <> "Healthy"',
                'format': injured_format
            })


    print(f"\n✅ Successfully exported all data to: {full_output_path}")

if __name__ == "__main__":
    pd.set_option('display.max_rows', 200); pd.set_option('display.max_columns', None); pd.set_option('display.width', 1000)
    run_daily_projections()

Mounted at /content/drive
--- Fetching Live Injury Data from ESPN ---
Successfully fetched injury status for 93 players.

--- Loading Trained Models & Performance Data ---

--- Step 1: Fetching today's game schedule from Odds API ---

--- Step 2: Acquiring Player Rosters & Positions (with self-healing) ---
Master scraping attempt 1/3...
--- Scraping B-Ref rosters for 22 teams for season 2026 ---
  - CRITICAL: Scrape attempt 1 failed with an unexpected error: Message: session not created: cannot connect to chrome at 127.0.0.1:47069
from chrome not reachable; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x58c04527eaca <unknown>
#1 0x58c044ccab08 <unknown>
#2 0x58c044cb6796 <unknown>
#3 0x58c044d0b36c <unknown>
#4 0x58c044d00ab9 <unknown>
#5 0x58c044d51eb6 <unknown>
#6 0x58c044d515d6 <unknown>
#7 0x58c044d0faca <unknown>
#8 0x58c044d107d1 <unknown>
#9 0x58c045245a99 <unknown

KeyboardInterrupt: 

In [ ]:
# ==============================================================================
# SCRIPT 3: The Definitive Prediction & Multi-Sheet Output Engine (FINALIZED)
# ==============================================================================

# ==============================================================================
# Step 0: Install all required packages
# ==============================================================================
!pip install selenium xlsxwriter nba_api scikit-learn==1.3.0 pandas joblib numpy==1.26.4 pytz unidecode requests lxml tqdm
!apt-get update --quiet
!apt-get install -y chromium-chromedriver --quiet

# ==============================================================================
# Step 1: Import all necessary libraries
# ==============================================================================
import pandas as pd
import numpy as np
import requests
import re
import time
import os
import joblib
import pytz
import glob
import json
from datetime import datetime, timezone, timedelta
from unidecode import unidecode
from io import StringIO
from google.colab import drive
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, playergamelogs, leaguedashplayerstats, drafthistory, teamestimatedmetrics
from IPython.display import display
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from xlsxwriter.utility import xl_col_to_name
from tqdm import tqdm
import signal
from functools import wraps


# ==============================================================================
# --- CONFIGURATION ---
# ==============================================================================
API_TIMEOUT = 120
API_RETRY_ATTEMPTS = 3
API_RETRY_DELAY = 3
INTER_PLAYER_DELAY = 0.6
WEB_SCRAPE_DELAY = 5
WEB_SCRAPE_RETRY_DELAY = 20
MASTER_SCRAPE_ATTEMPTS = 3
MASTER_SCRAPE_TIMEOUT = 600

ODDS_API_KEYS = [
    "9b96859c5e0781bb4c5809215a5ed94b", "f4cfcea7cc6120a97559f69d6049596e", "7b9405e2febc561e554b7169a42cefd1",
    "146dff42ae19a83208b059d4855095c9", "f8474479351ccc685ccae9f4716c23c5", "39756c093848b6968c899a4e1ded85f9",
    "9705a1f6b609305047eb8ec9243cc54a", "4217574ccdd90732483d98f24772f66b", "66953d96fcb2ec72a47e7f39901258e7",
    "94304f58fb95fd1d45159b9c8d292724", "e384411e342cac5c9408804c622703f1", "dab5c3da80b0415f64d4d94890974cdf",
    "9705a1f6b609305047eb8ec9243cc54a", "3aae94b51cc3dcb8e0623bf2633a695a", "a4e4d3eef7d360055b96a1351560817f",
    "bd6c88c66ff4c7c057aafaffcdd1b8d", "1bfcac154fb2b490bf55a80a18275db1", "7b64d450f91079f60211cb33033594b4",
    "ec9dc1e1aa0543c1b3fa3d2576fabffd", "6bbfe433274aef6fee5aa5543ad69055", "1f5e9cdc190854c605b533917b2c9b82",
    "858b7234b5f77661345f205b1c5ec121"
]
CONFIG = { "GAMES_PLAYED_THRESHOLD_ACCLIMATED": 15, "GAMES_PLAYED_THRESHOLD_LOW_VOLUME": 20 }

# ==============================================================================
# Part 2: Helper Functions and Data Fetching
# ==============================================================================
class TimeoutError(Exception): pass
def run_with_timeout(seconds):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            def handler(signum, frame): raise TimeoutError(f"Function '{func.__name__}' timed out")
            signal.signal(signal.SIGALRM, handler)
            signal.alarm(seconds)
            try: result = func(*args, **kwargs)
            finally: signal.alarm(0)
            return result
        return wrapper
    return decorator

def normalize_player_name(name):
    if not isinstance(name, str): return None
    name = unidecode(name.lower()); name = re.sub(r'\s+(jr|sr|ii|iii|iv)\.?$', '', name); name = re.sub(r'[^\w\s]', '', name); name = re.sub(r'\s+', ' ', name).strip()
    return name

def get_implied_prob_from_decimal(odd):
    if odd is None or pd.isna(odd) or float(odd) < 1: return None
    return 1 / float(odd)

def decimal_to_american(decimal_odds):
    if pd.isna(decimal_odds) or decimal_odds is None: return None
    if decimal_odds >= 2.00: return f"+{int((decimal_odds - 1) * 100)}"
    else: return f"-{int(100 / (decimal_odds - 1))}"

def get_current_nba_season():
    now = datetime.now()
    if now.month >= 8: start_year, end_year = now.year, now.year + 1
    else: end_year, start_year = now.year, now.year - 1
    return f"{start_year}-{str(end_year)[-2:]}"

@run_with_timeout(MASTER_SCRAPE_TIMEOUT)
def scrape_bref_positions_by_team(season_end_year, teams_to_scrape):
    print(f"--- Scraping B-Ref rosters for {len(teams_to_scrape)} teams for season {season_end_year} ---")
    all_players_df_list = []
    chrome_options = Options(); chrome_options.add_argument('--headless'); chrome_options.add_argument('--no-sandbox'); chrome_options.add_argument('--disable-dev-shm-usage')
    driver = webdriver.Chrome(options=chrome_options)
    try:
        for team_abbr_bref in tqdm(teams_to_scrape, desc="Scraping B-Ref Positions"):
            for attempt in range(API_RETRY_ATTEMPTS):
                try:
                    url = f"https://www.basketball-reference.com/teams/{team_abbr_bref}/{season_end_year}.html"
                    driver.get(url)
                    WebDriverWait(driver, 30).until(EC.presence_of_element_located((By.ID, "roster")))
                    tables = pd.read_html(StringIO(driver.page_source), attrs={'id': 'roster'})
                    if tables:
                        team_df = tables[0].copy()
                        team_df['CURRENT_TEAM_BRef'] = team_abbr_bref
                        team_df.rename(columns={'Player': 'BRef_Name', 'Pos': 'DetailedPos'}, inplace=True)
                        all_players_df_list.append(team_df)
                    time.sleep(WEB_SCRAPE_DELAY)
                    break
                except Exception as e:
                    print(f"  - Attempt {attempt+1} failed for {team_abbr_bref}: {e}")
                    if attempt < API_RETRY_ATTEMPTS - 1:
                        time.sleep(WEB_SCRAPE_RETRY_DELAY)
                    else:
                        print(f"  - CRITICAL: All attempts failed for {team_abbr_bref}. Skipping team.")
    finally:
        driver.quit()
    if not all_players_df_list: print("CRITICAL ERROR: Failed to scrape any team data."); return pd.DataFrame()
    return pd.concat(all_players_df_list, ignore_index=True)

def update_and_get_live_dvp(season, data_dir, bref_positions_df):
    print("\n--- Updating and Calculating Live DvP Data ---")
    gamelogs_cache_path = os.path.join(data_dir, f"live_gamelogs_{season}.csv")
    existing_gamelogs = pd.DataFrame()
    last_date_in_cache = None
    if os.path.exists(gamelogs_cache_path):
        try:
            existing_gamelogs = pd.read_csv(gamelogs_cache_path, low_memory=False)
            existing_gamelogs['GAME_DATE'] = pd.to_datetime(existing_gamelogs['GAME_DATE'])
            if not existing_gamelogs.empty:
                last_date_in_cache = existing_gamelogs['GAME_DATE'].max()
                print(f"Loaded {len(existing_gamelogs)} cached gamelogs. Last entry is from: {last_date_in_cache.date()}")
        except Exception as e: print(f"Warning: Could not read gamelogs cache. Error: {e}")

    date_from = (last_date_in_cache + timedelta(days=1)).strftime('%m/%d/%Y') if last_date_in_cache else None

    try:
        print(f"Fetching new game logs from {date_from or 'the beginning of the season'}...")
        new_gamelogs = playergamelogs.PlayerGameLogs(season_nullable=season, date_from_nullable=date_from, timeout=API_TIMEOUT).get_data_frames()[0]
        if not new_gamelogs.empty:
            new_gamelogs['GAME_DATE'] = pd.to_datetime(new_gamelogs['GAME_DATE'])
            all_gamelogs = pd.concat([existing_gamelogs, new_gamelogs], ignore_index=True)
            all_gamelogs.drop_duplicates(subset=['GAME_ID', 'PLAYER_ID'], keep='last', inplace=True)
            all_gamelogs.to_csv(gamelogs_cache_path, index=False)
            print(f"Found {len(new_gamelogs)} new game logs. Cache updated.")
        else:
            print("No new game logs found. DvP data is up to date.")
            all_gamelogs = existing_gamelogs
    except Exception as e:
        print(f"Warning: Failed to fetch new game logs. Using cached data. Error: {e}")
        all_gamelogs = existing_gamelogs

    if all_gamelogs.empty: print("CRITICAL: No game logs available to calculate DvP."); return pd.DataFrame()

    print("Calculating season-to-date DvP stats...")
    if bref_positions_df.empty: print("CRITICAL WARNING: No position data from B-Ref. DvP will be inaccurate."); return pd.DataFrame()

    bref_positions_df['BRef_Name_Normalized'] = bref_positions_df['BRef_Name'].apply(normalize_player_name)
    all_gamelogs['PLAYER_NAME_NORMALIZED'] = all_gamelogs['PLAYER_NAME'].apply(normalize_player_name)
    gamelogs_with_pos = pd.merge(all_gamelogs, bref_positions_df[['BRef_Name_Normalized', 'DetailedPos']], left_on='PLAYER_NAME_NORMALIZED', right_on='BRef_Name_Normalized', how='left')
    gamelogs_with_pos.rename(columns={'DetailedPos': 'Game_Position'}, inplace=True)
    gamelogs_with_pos['Game_Position'] = gamelogs_with_pos['Game_Position'].apply(lambda x: x.split('-')[0] if isinstance(x, str) and x else 'Unknown')

    opponent_df = gamelogs_with_pos[['GAME_ID', 'TEAM_ABBREVIATION']].drop_duplicates()
    gamelogs_with_pos = pd.merge(gamelogs_with_pos, opponent_df.rename(columns={'TEAM_ABBREVIATION': 'OPPONENT_ABBR'}), on='GAME_ID')
    gamelogs_with_pos = gamelogs_with_pos[gamelogs_with_pos['TEAM_ABBREVIATION'] != gamelogs_with_pos['OPPONENT_ABBR']]
    for stat in ['PTS', 'REB', 'AST']: gamelogs_with_pos[stat] = pd.to_numeric(gamelogs_with_pos[stat], errors='coerce')
    dvp = gamelogs_with_pos.groupby(['OPPONENT_ABBR', 'Game_Position'])[['PTS', 'REB', 'AST']].mean().reset_index()
    dvp_pivot = dvp.pivot_table(index='OPPONENT_ABBR', columns='Game_Position', values=['PTS', 'REB', 'AST'])
    dvp_pivot.columns = [f'OPP_DvP_{stat}_vs_{pos}' for stat, pos in dvp_pivot.columns]
    print("--- DvP Calculation Complete ---")
    return dvp_pivot

feature_cache = {}
def get_live_features_for_player(player_id, stats_season):
    if player_id in feature_cache: return feature_cache[player_id]
    print(f"  - Engineering features for player ID: {player_id}...")
    try:
        logs = playergamelog.PlayerGameLog(player_id=player_id, season=stats_season, timeout=API_TIMEOUT).get_data_frames()[0]
        if logs.empty or len(logs) < 3: return None
        logs['GAME_DATE'] = pd.to_datetime(logs['GAME_DATE']); logs = logs.sort_values('GAME_DATE', ascending=True)
        features = {}
        stats_to_roll = ['PTS', 'REB', 'AST', 'MIN', 'FGA', 'FTA', 'FG3A']
        windows = [3, 5, 10]
        for stat in stats_to_roll:
            for w in windows: features[f'ROLL_AVG_{stat}_{w}'] = logs[stat].tail(w).mean()

        # --- CORRECTED: Backfill missing longer-term averages with shorter-term ones in the correct order ---
        for stat in stats_to_roll:
            # First, ensure 5-game avg is filled if it's missing
            if pd.isna(features.get(f'ROLL_AVG_{stat}_5')):
                features[f'ROLL_AVG_{stat}_5'] = features.get(f'ROLL_AVG_{stat}_3')
            # Now, ensure 10-game avg is filled with the (potentially newly filled) 5-game avg
            if pd.isna(features.get(f'ROLL_AVG_{stat}_10')):
                features[f'ROLL_AVG_{stat}_10'] = features.get(f'ROLL_AVG_{stat}_5')

        features['DAYS_OF_REST'] = min((datetime.now(timezone.utc).replace(tzinfo=None) - logs.iloc[-1]['GAME_DATE']).days, 7)
        feature_cache[player_id] = features
        return features
    except Exception as e:
        print(f"    - Could not fetch game logs for player {player_id}. Error: {e}"); return None

def get_draft_class_ids(season_year):
    print(f"--- Fetching {season_year} Draft Class ---")
    try:
        draft_history_data = drafthistory.DraftHistory(season_year_nullable=season_year)
        df_draft = draft_history_data.get_data_frames()[0]
        return set(df_draft['PERSON_ID'].unique())
    except Exception as e:
        print(f"Could not fetch draft class for {season_year}: {e}"); return set()

def identify_team_changes(master_df):
    print("--- Identifying Players on New Teams ---")
    changed_teams_df = master_df[(master_df['TEAM_ABBREVIATION'] != master_df['CURRENT_TEAM']) & (master_df['CURRENT_TEAM'].notna()) & (master_df['TEAM_ABBREVIATION'].notna())]
    return set(changed_teams_df['PLAYER_ID'].unique())

def get_player_experience(row, rookie_ids, traded_player_ids):
    player_id, games_played_last_season, games_played_this_season = row['PLAYER_ID'], row.get('GP', 99), row.get('GP_current', 0)
    is_rookie = player_id in rookie_ids
    is_traded = player_id in traded_player_ids
    is_acclimated = games_played_this_season >= CONFIG["GAMES_PLAYED_THRESHOLD_ACCLIMATED"]
    if is_rookie: return "Traded Rookie" if is_traded and not is_acclimated else "Rookie"
    else:
        if is_traded and not is_acclimated: return "Traded Veteran"
        if games_played_last_season < CONFIG["GAMES_PLAYED_THRESHOLD_LOW_VOLUME"]: return "Low Volume"
        return "Veteran"

# ==============================================================================
# Part 3: The Main Prediction Engine
# ==============================================================================
def run_daily_projections():
    try:
        drive.mount('/content/drive', force_remount=True)
        DATA_DIR = "/content/drive/MyDrive/NBA/Historical Data/"
        MODEL_DIR = "/content/drive/MyDrive/NBA/Training Model"
        OUTPUT_DIR = "/content/drive/MyDrive/NBA/Predictions"
        os.makedirs(MODEL_DIR, exist_ok=True); os.makedirs(OUTPUT_DIR, exist_ok=True); os.makedirs(DATA_DIR, exist_ok=True)
    except Exception as e: print(f"Could not mount Google Drive. Aborting. Error: {e}"); return

    print("\n--- Loading Trained Models & Performance Data ---")
    regression_props = ['PTS', 'REB', 'AST', 'P+R', 'P+A', 'R+A', 'PRA']; classification_props = ['DD', 'TD']
    models = {}; mae_values = {}
    try:
        for prop in regression_props: models[prop] = joblib.load(os.path.join(MODEL_DIR, f"nba_{prop.lower()}_ridge_model.pkl"))
        for prop in classification_props: models[prop] = joblib.load(os.path.join(MODEL_DIR, f"nba_{prop.lower()}_logreg_model.pkl"))
        with open(os.path.join(MODEL_DIR, "mae_values.json"), 'r') as f: mae_values = json.load(f)
    except FileNotFoundError as e: print(f"CRITICAL ERROR: Could not load model files. Please run Script 2. Missing: {e}"); return

    CURRENT_SEASON = get_current_nba_season()
    eastern = pytz.timezone('US/Eastern'); now_et = datetime.now(eastern)

    print("\n--- Step 1: Fetching today's game schedule from Odds API ---")
    start_of_day_et = now_et.replace(hour=8, minute=0, second=0, microsecond=0); end_of_day_et = start_of_day_et + timedelta(days=1)
    start_utc_str = start_of_day_et.astimezone(pytz.utc).strftime('%Y-%m-%dT%H:%M:%SZ'); end_utc_str = end_of_day_et.astimezone(pytz.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    working_key, events = None, []
    for i, api_key in enumerate(ODDS_API_KEYS):
        try:
            response = requests.get('https://api.the-odds-api.com/v4/sports/basketball_nba/events', params={'apiKey': api_key, 'commenceTimeFrom': start_utc_str, 'commenceTimeTo': end_utc_str}, timeout=30)
            response.raise_for_status(); events = response.json(); working_key = api_key; break
        except Exception as e: print(f"--- API Key #{i+1} failed. Error: {e} ---")
    if not working_key: print("--- CRITICAL: All API keys failed. ---"); return
    if not events: print("No NBA games scheduled for today."); return
    team_name_map = {team['full_name']: team['abbreviation'] for team in teams.get_teams()}
    games_today = [{'home_team_abbr': team_name_map.get(e['home_team']), 'away_team_abbr': team_name_map.get(e['away_team'])} for e in events if team_name_map.get(e['home_team']) and team_name_map.get(e['away_team'])]
    games_today = [dict(t) for t in {tuple(d.items()) for d in games_today}]

    print("\n--- Step 2: Acquiring Player Rosters & Positions (with self-healing) ---")
    today_str = now_et.strftime('%Y-%m-%d')
    current_year_start = int(CURRENT_SEASON.split('-')[0])
    POSITIONS_SEASON_END_YEAR = current_year_start + 1
    teams_playing_today = {g['home_team_abbr'] for g in games_today} | {g['away_team_abbr'] for g in games_today}
    bref_team_map = {"BKN": "BRK", "PHX": "PHO", "CHA": "CHO"}; teams_to_scrape_bref = [bref_team_map.get(team, team) for team in teams_playing_today]

    bref_df = pd.DataFrame()
    for attempt in range(MASTER_SCRAPE_ATTEMPTS):
        try:
            print(f"Master scraping attempt {attempt + 1}/{MASTER_SCRAPE_ATTEMPTS}...")
            bref_df = scrape_bref_positions_by_team(season_end_year=POSITIONS_SEASON_END_YEAR, teams_to_scrape=teams_to_scrape_bref)
            if not bref_df.empty and set(bref_df['CURRENT_TEAM_BRef'].unique()) == set(teams_to_scrape_bref):
                print("✅ Successfully scraped all required team rosters.")
                break
            else:
                print(f"  - Warning: Scrape attempt {attempt+1} was incomplete. Retrying...")
        except TimeoutError as e:
            print(f"  - CRITICAL: Scrape attempt {attempt+1} timed out. Error: {e}")
        except Exception as e:
            print(f"  - CRITICAL: Scrape attempt {attempt+1} failed with an unexpected error: {e}")
        if attempt < MASTER_SCRAPE_ATTEMPTS - 1:
            print("    -> Waiting before next master attempt...")
            time.sleep(WEB_SCRAPE_RETRY_DELAY)

    if bref_df.empty:
        print("CRITICAL FAILURE: Could not scrape rosters after multiple master attempts. Aborting."); return

    dvp_df = update_and_get_live_dvp(CURRENT_SEASON, DATA_DIR, bref_df)

    print("\n--- Step 3: Fetching player prop odds from Odds API ---")
    MARKETS_TO_KEEP = {'player_points': 'PTS', 'player_rebounds': 'REB', 'player_assists': 'AST', 'player_points_rebounds_assists': 'PRA', 'player_points_rebounds': 'P+R', 'player_points_assists': 'P+A', 'player_rebounds_assists': 'R+A', 'player_double_double': 'DD', 'player_triple_double': 'TD'}
    all_props = [];
    key_usage = {}
    print("Checking API key usage...")
    for key in ODDS_API_KEYS:
        try:
            usage_response = requests.get('https://api.the-odds-api.com/v4/sports', params={'apiKey': key})
            remaining = int(usage_response.headers.get('x-requests-remaining', 0))
            key_usage[key] = remaining
            print(f"  - Key ending in ...{key[-4:]} has {remaining} requests remaining.")
        except Exception as e:
            key_usage[key] = 0
            print(f"  - Could not check usage for key ending in ...{key[-4:]}. Error: {e}")

    current_key_index = 0
    for event in events:
        event_id = event['id']
        odds_fetched = False
        while not odds_fetched and current_key_index < len(ODDS_API_KEYS):
            api_key = ODDS_API_KEYS[current_key_index]
            if key_usage.get(api_key, 0) < len(events):
                print(f"Skipping key ...{api_key[-4:]}, not enough requests remaining.")
                current_key_index += 1
                continue
            params = {'apiKey': api_key, 'regions': 'us', 'markets': ','.join(MARKETS_TO_KEEP.keys()), 'bookmakers': 'draftkings,fanduel'}
            try:
                response = requests.get(f'https://api.the-odds-api.com/v4/sports/basketball_nba/events/{event_id}/odds', params=params, timeout=30)
                response.raise_for_status()
                for bookmaker in response.json().get('bookmakers', []):
                    for market in bookmaker.get('markets', []):
                        prop_type = MARKETS_TO_KEEP.get(market['key'])
                        if prop_type:
                            for outcome in market.get('outcomes', []):
                                all_props.append({ 'Game': f"{team_name_map.get(event['away_team'])} @ {team_name_map.get(event['home_team'])}", 'Game_Time': event['commence_time'], 'Player': outcome['description'], 'Prop': prop_type, 'Line': outcome.get('point'), 'Over_Odds': outcome['price'] if outcome['name'] == 'Over' else None, 'Under_Odds': outcome['price'] if outcome['name'] == 'Under' else None, 'Yes_Odds': outcome['price'] if outcome['name'] == 'Yes' else None, 'No_Odds': outcome['price'] if outcome['name'] == 'No' else None, 'Bookmaker': bookmaker['title'] })
                odds_fetched = True
            except requests.exceptions.RequestException as e:
                print(f"Error fetching odds for event {event_id}. Key #{current_key_index + 1} failed. Error: {e}")
                current_key_index += 1
        if not odds_fetched and current_key_index >= len(ODDS_API_KEYS):
            print("All Odds API keys failed. Aborting odds fetch for remaining games.")
            break

    if not all_props: print("No player props were available. Aborting."); return
    df = pd.DataFrame(all_props); df['match_key'] = df['Player'].apply(normalize_player_name)
    odds_df_ou = df[df['Prop'].isin(regression_props)].groupby(['match_key', 'Player', 'Prop', 'Line', 'Bookmaker', 'Game', 'Game_Time']).agg({'Over_Odds': 'first', 'Under_Odds': 'first'}).reset_index()
    odds_df_yn = df[df['Prop'].isin(classification_props)].groupby(['match_key', 'Player', 'Prop', 'Bookmaker', 'Game', 'Game_Time']).agg({'Yes_Odds': 'first', 'No_Odds': 'first'}).reset_index()
    odds_df_yn['Implied_Prob_Yes'] = odds_df_yn['Yes_Odds'].apply(get_implied_prob_from_decimal)
    odds_df = pd.concat([odds_df_ou, odds_df_yn], ignore_index=True)
    for col in ['Over_Odds', 'Under_Odds', 'Yes_Odds', 'No_Odds']: odds_df[col] = odds_df[col].apply(decimal_to_american)
    odds_df['Game_Time'] = pd.to_datetime(odds_df['Game_Time']).dt.tz_convert('US/Eastern').dt.strftime('%-I:%M %p EST')

    print("\n--- Step 4: Preparing Data for Projections ---")
    use_previous_season_stats = today_str < f"{current_year_start}-12-01"
    STATS_SEASON_STRING = f"{current_year_start - 1}-{str(current_year_start)[-2:]}" if use_previous_season_stats else CURRENT_SEASON
    nba_df = leaguedashplayerstats.LeagueDashPlayerStats(season=STATS_SEASON_STRING, per_mode_detailed="PerGame").get_data_frames()[0]
    nba_df_current = leaguedashplayerstats.LeagueDashPlayerStats(season=CURRENT_SEASON, per_mode_detailed="PerGame").get_data_frames()[0] if use_previous_season_stats and CURRENT_SEASON != STATS_SEASON_STRING else nba_df

    nba_df['merge_key'] = nba_df['PLAYER_NAME'].apply(normalize_player_name)
    reverse_bref_map = {v: k for k, v in bref_team_map.items()}
    bref_df['CURRENT_TEAM'] = bref_df['CURRENT_TEAM_BRef'].apply(lambda x: reverse_bref_map.get(x, x))
    bref_df['merge_key'] = bref_df['BRef_Name'].apply(normalize_player_name)
    master_df = pd.merge(nba_df, bref_df, on='merge_key', how='inner')

    if use_previous_season_stats: master_df['GP_current'] = master_df['PLAYER_ID'].map(nba_df_current.set_index('PLAYER_ID')['GP']).fillna(0)
    else: master_df['GP_current'] = master_df['GP']
    rookie_ids = get_draft_class_ids(str(current_year_start))
    traded_player_ids = identify_team_changes(master_df)

    players_with_odds = odds_df['match_key'].unique()
    players_to_project_df = master_df[master_df['merge_key'].isin(players_with_odds)]
    print(f"\n--- Step 5: Generating Projections for {len(players_to_project_df)} Players with Active Props ---")

    all_projections = []
    for _, player in players_to_project_df.iterrows():
        time.sleep(INTER_PLAYER_DELAY)
        player_features = get_live_features_for_player(player['PLAYER_ID'], CURRENT_SEASON)
        if not player_features: continue

        opp_team_abbr = next((g['home_team_abbr'] if player['CURRENT_TEAM'] == g['away_team_abbr'] else g['away_team_abbr'] for g in games_today if player['CURRENT_TEAM'] in [g['home_team_abbr'], g['away_team_abbr']]), None)
        if not opp_team_abbr: continue

        player_pos = player['DetailedPos'].split('-')[0]
        for stat in ['PTS', 'REB', 'AST']:
            dvp_feature_name = f'OPP_DvP_{stat}_vs_POS'
            dvp_lookup_col = f'OPP_DvP_{stat}_vs_{player_pos}'
            if not dvp_df.empty and opp_team_abbr in dvp_df.index and dvp_lookup_col in dvp_df.columns:
                player_features[dvp_feature_name] = dvp_df.loc[opp_team_abbr, dvp_lookup_col]
            else:
                player_features[dvp_feature_name] = np.nan

        projections_for_player = {'Player': player['PLAYER_NAME'], 'Team': player['CURRENT_TEAM'], 'Pos': player['DetailedPos'], 'match_key': player['merge_key'], 'Experience': get_player_experience(player, rookie_ids, traded_player_ids)}

        TRAINING_FEATURES = [
            'ROLL_AVG_PTS_3', 'ROLL_AVG_REB_3', 'ROLL_AVG_AST_3', 'ROLL_AVG_PTS_5',
            'ROLL_AVG_REB_5', 'ROLL_AVG_AST_5', 'ROLL_AVG_PTS_10', 'ROLL_AVG_REB_10',
            'ROLL_AVG_AST_10', 'ROLL_AVG_MIN_10', 'ROLL_AVG_FGA_10', 'ROLL_AVG_FTA_10',
            'ROLL_AVG_FG3A_10', 'DAYS_OF_REST',
            'OPP_DvP_PTS_vs_POS', 'OPP_DvP_REB_vs_POS', 'OPP_DvP_AST_vs_POS'
        ]
        feature_df = pd.DataFrame([player_features])
        feature_df = feature_df[TRAINING_FEATURES]

        for prop in regression_props: projections_for_player[f'Proj_{prop}'] = round(max(0, models[prop].predict(feature_df)[0]), 1)
        for prop in classification_props: projections_for_player[f'Proj_{prop}_%'] = f"{models[prop].predict_proba(feature_df)[0][1]:.1%}"
        all_projections.append(projections_for_player)

    if not all_projections: print("\n--- WARNING: No projections were generated. ---"); return
    projections_df = pd.DataFrame(all_projections)

    print("\n--- Step 6: Calculating Edge and Generating Output Files ---")
    melted_projections = projections_df.melt(id_vars=['match_key', 'Player', 'Team', 'Pos', 'Experience'], value_vars=[c for c in projections_df.columns if c.startswith('Proj')], var_name='Prop_Name', value_name='Projection')
    melted_projections['Prop'] = melted_projections['Prop_Name'].str.replace('Proj_', '').str.replace('_%', '')
    master_long_df = pd.merge(melted_projections, odds_df, on=['match_key', 'Prop'], how='left', suffixes=('_proj', '_odds'))
    master_long_df.dropna(subset=['Game'], inplace=True)
    master_long_df['Player'] = master_long_df['Player_proj'].fillna(master_long_df['Player_odds']); master_long_df.drop(columns=['Player_proj', 'Player_odds'], inplace=True, errors='ignore')
    master_long_df['Projection_Num'] = pd.to_numeric(master_long_df['Projection'].astype(str).str.replace('%', ''), errors='coerce')
    master_long_df.drop(columns=['Projection'], inplace=True, errors='ignore')

    yn_mask = master_long_df['Prop'].isin(classification_props)
    ou_mask = master_long_df['Prop'].isin(regression_props)

    master_long_df['DD & TD\nModel Score'] = np.nan
    master_long_df.loc[yn_mask, 'DD & TD\nModel Score'] = master_long_df.loc[yn_mask, 'Projection_Num']
    master_long_df.loc[yn_mask, 'Projection_Num'] = np.nan

    for prop, mae in mae_values.items():
        mask = master_long_df['Prop'] == prop
        master_long_df.loc[mask, 'Proj Range (Low)'] = (master_long_df.loc[mask, 'Projection_Num'] - mae).round(1).apply(lambda x: max(0, x))
        master_long_df.loc[mask, 'Proj Range (High)'] = (master_long_df.loc[mask, 'Projection_Num'] + mae).round(1)
    master_long_df['Proj Range'] = master_long_df.apply(lambda row: f"{row['Proj Range (Low)']}-{row['Proj Range (High)']}" if pd.notna(row['Proj Range (Low)']) else "", axis=1)

    master_long_df['Edge'] = None; master_long_df['Bet'] = 'Pass'
    master_long_df.loc[ou_mask, 'Edge'] = ((master_long_df.loc[ou_mask, 'Projection_Num'] - master_long_df.loc[ou_mask, 'Line']) / master_long_df.loc[ou_mask, 'Line']) * 100
    master_long_df.loc[ou_mask, 'Bet'] = master_long_df.loc[ou_mask, 'Edge'].apply(lambda x: 'Over' if pd.notna(x) and x > 10 else ('Under' if pd.notna(x) and x < -10 else 'Pass'))
    if 'Implied_Prob_Yes' in master_long_df.columns:
        master_long_df.loc[yn_mask, 'Edge'] = master_long_df.loc[yn_mask, 'DD & TD\nModel Score'] - (master_long_df.loc[yn_mask, 'Implied_Prob_Yes'].fillna(0) * 100)
    else:
        master_long_df.loc[yn_mask, 'Edge'] = None
    master_long_df.loc[yn_mask, 'Bet'] = master_long_df.loc[yn_mask, 'Edge'].apply(lambda x: 'Yes' if pd.notna(x) and x > 5 else ('No' if pd.notna(x) and x < -5 else 'Pass'))
    if 'Implied_Prob_Yes' in master_long_df.columns: master_long_df.drop(columns=['Implied_Prob_Yes'], inplace=True)


    master_long_df['Alt Line'] = 'Pass'
    alt_line_mask = (master_long_df['Bet'] == 'Over') & (master_long_df['Proj Range (Low)'].notna())
    alt_line_values = np.floor(pd.to_numeric(master_long_df.loc[alt_line_mask, 'Proj Range (Low)'], errors='coerce'))
    valid_alt_line_mask = alt_line_values < master_long_df.loc[alt_line_mask, 'Line']
    master_long_df.loc[alt_line_mask & valid_alt_line_mask, 'Alt Line'] = 'Over ' + alt_line_values[valid_alt_line_mask].astype(int).astype(str)

    master_long_df['DD/TD Odds'] = master_long_df['Yes_Odds']
    master_long_df.rename(columns={'Projection_Num': 'Projection', 'Over_Odds': 'Over Odds', 'Under_Odds': 'Under Odds'}, inplace=True)
    master_long_df.drop(columns=['Proj Range (Low)', 'Proj Range (High)', 'Yes_Odds', 'No_Odds'], inplace=True, errors='ignore')

    cols_to_round = ['DD & TD\nModel Score', 'Line', 'Projection', 'Edge']
    for col in cols_to_round:
        if col in master_long_df.columns:
            master_long_df[col] = pd.to_numeric(master_long_df[col], errors='coerce').round(1)

    game_date_str = pd.to_datetime(events[0]
    ['commence_time']).tz_convert('US/Eastern').strftime('%m-%d-%Y')
    output_filename = f"{game_date_str} NBA Props Predictions.xlsx"
    full_output_path = os.path.join(OUTPUT_DIR, output_filename)
    with pd.ExcelWriter(full_output_path, engine='xlsxwriter') as writer:
        workbook  = writer.book
        header_format = workbook.add_format({'bold': True, 'align': 'center', 'valign': 'vcenter', 'fg_color': '#D7E4BC', 'border': 1, 'text_wrap': True})
        center_format = workbook.add_format({'align': 'center', 'valign': 'vcenter', 'border': 1})

        ev_cols = ['Game_Time', 'Game', 'Player', 'Team', 'Prop', 'DD & TD\nModel Score', 'Line', 'Projection', 'Proj Range', 'Alt Line', 'Edge', 'Bet', 'Over Odds', 'Under Odds', 'DD/TD Odds']
        final_cols = ['Player', 'Team', 'Prop', 'DD & TD\nModel Score', 'Line', 'Projection', 'Proj Range', 'Alt Line', 'Edge', 'Bet', 'Over Odds', 'Under Odds', 'DD/TD Odds', 'Bookmaker']

        all_ev_bets_df = master_long_df[master_long_df['Bet'] != 'Pass'].copy()
        if not all_ev_bets_df.empty:
            for bookmaker in ['DraftKings', 'FanDuel']:
                book_df = all_ev_bets_df[all_ev_bets_df['Bookmaker'] == bookmaker]
                if book_df.empty: continue
                sorted_games = sorted(games_today, key=lambda g: (g['home_team_abbr'], g['away_team_abbr']))
                game_dfs = [game_df.sort_values(by='Edge', key=abs, ascending=False) for game_tuple in sorted_games if not (game_df := book_df[book_df['Game'] == f"{game_tuple['away_team_abbr']} @ {game_tuple['home_team_abbr']}"]).empty]
                if game_dfs:
                    final_ev_df = pd.concat(game_dfs, ignore_index=True)
                    final_ev_df = final_ev_df.reindex(columns=ev_cols)
                    sheet_name = f"{bookmaker}_EV_Bets"
                    final_ev_df.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=1)
                    worksheet = writer.sheets[sheet_name]

                    for col_num, value in enumerate(final_ev_df.columns.values):
                        worksheet.write(0, col_num, value, header_format)
                        data_len = final_ev_df[value].astype(str).map(len).max() if not final_ev_df[value].isnull().all() else 0
                        header_len = max([len(s) for s in value.split('\n')]) if '\n' in value else len(value)
                        column_width = max(data_len, header_len, 10) + 2
                        worksheet.set_column(col_num, col_num, column_width, center_format)
                    worksheet.set_row(0, 30)

        for game in sorted(games_today, key=lambda g: (g['home_team_abbr'], g['away_team_abbr'])):
            home_abbr, away_abbr = game['home_team_abbr'], game['away_team_abbr']
            sheet_name = f"{away_abbr}_vs_{home_abbr}_Full_Data"
            game_df = master_long_df[master_long_df['Game'] == f"{away_abbr} @ {home_abbr}"].copy()
            if game_df.empty: continue
            game_long_df = game_df.reindex(columns=final_cols).sort_values(by=['Player', 'Prop'])
            game_long_df.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=1)
            worksheet = writer.sheets[sheet_name]

            for col_num, value in enumerate(game_long_df.columns.values):
                worksheet.write(0, col_num, value, header_format)
                data_len = game_long_df[value].astype(str).map(len).max() if not game_long_df[value].isnull().all() else 0
                header_len = max([len(s) for s in value.split('\n')]) if '\n' in value else len(value)
                column_width = max(data_len, header_len, 10) + 2
                worksheet.set_column(col_num, col_num, column_width, center_format)
            worksheet.set_row(0, 30)

    print(f"\n✅ Successfully exported all data to: {full_output_path}")

if __name__ == "__main__":
    pd.set_option('display.max_rows', 200); pd.set_option('display.max_columns', None); pd.set_option('display.width', 1000)
    run_daily_projections()

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://cli.github.com/packages stable InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:12 https://dl.google.com/linux/chrome/deb stable InRelease
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem t

Scraping B-Ref Positions:   0%|          | 0/22 [00:00<?, ?it/s]

  - Attempt 1 failed for SAS: Message: 
Stacktrace:
#0 0x5b5fd8403aca <unknown>
#1 0x5b5fd7e4fcdb <unknown>
#2 0x5b5fd7ea26c4 <unknown>
#3 0x5b5fd7ea2901 <unknown>
#4 0x5b5fd7ef18b4 <unknown>
#5 0x5b5fd7eeec87 <unknown>
#6 0x5b5fd7e94aca <unknown>
#7 0x5b5fd7e957d1 <unknown>
#8 0x5b5fd83caa99 <unknown>
#9 0x5b5fd83cda6c <unknown>
#10 0x5b5fd83b3d29 <unknown>
#11 0x5b5fd83ce665 <unknown>
#12 0x5b5fd839b6a3 <unknown>
#13 0x5b5fd83f07b8 <unknown>
#14 0x5b5fd83f0993 <unknown>
#15 0x5b5fd8402a63 <unknown>
#16 0x7fea9aa2dac3 <unknown>



Scraping B-Ref Positions:   0%|          | 0/22 [01:10<?, ?it/s]


KeyboardInterrupt: 

Script 3: V3

In [ ]:
# ==============================================================================
# SCRIPT 3: The Definitive Prediction & Multi-Sheet Output Engine (FINALIZED)
# ==============================================================================

# ==============================================================================
# Step 0: Install all required packages
# ==============================================================================
!pip install selenium xlsxwriter nba_api scikit-learn==1.3.0 pandas joblib numpy==1.26.4 pytz unidecode requests lxml tqdm undetected-chromedriver

# ==============================================================================
# Step 1: Import all necessary libraries
# ==============================================================================
import pandas as pd
import numpy as np
import requests
import re
import time
import os
import joblib
import pytz
import glob
import json
from datetime import datetime, timezone, timedelta
from unidecode import unidecode
from io import StringIO
from google.colab import drive
import undetected_chromedriver as uc
from selenium.webdriver.chrome.options import Options
from nba_api.stats.static import teams, players
from nba_api.stats.endpoints import playergamelog, playergamelogs, leaguedashplayerstats, drafthistory, teamestimatedmetrics
from IPython.display import display
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from xlsxwriter.utility import xl_col_to_name
from tqdm import tqdm
import signal
from functools import wraps


# ==============================================================================
# --- CONFIGURATION ---
# ==============================================================================
API_TIMEOUT = 120
API_RETRY_ATTEMPTS = 3
API_RETRY_DELAY = 3
INTER_PLAYER_DELAY = 0.6
WEB_SCRAPE_DELAY = 5
WEB_SCRAPE_RETRY_DELAY = 20
MASTER_SCRAPE_ATTEMPTS = 3
MASTER_SCRAPE_TIMEOUT = 600
USE_BASKETBALL_REFERENCE = False

ODDS_API_KEYS = [
    "9b96859c5e0781bb4c5809215a5ed94b", "f4cfcea7cc6120a97559f69d6049596e", "7b9405e2febc561e554b7169a42cefd1",
    "146dff42ae19a83208b059d4855095c9", "f8474479351ccc685ccae9f4716c23c5", "39756c093848b6968c899a4e1ded85f9",
    "9705a1f6b609305047eb8ec9243cc54a", "4217574ccdd90732483d98f24772f66b", "66953d96fcb2ec72a47e7f39901258e7",
    "94304f58fb95fd1d45159b9c8d292724", "e384411e342cac5c9408804c622703f1", "dab5c3da80b0415f64d4d94890974cdf",
    "9705a1f6b609305047eb8ec9243cc54a", "3aae94b51cc3dcb8e0623bf2633a695a", "a4e4d3eef7d360055b96a1351560817f",
    "bd6c88c66ff4c7c057aafaffcdd1b8d", "1bfcac154fb2b490bf55a80a18275db1", "7b64d450f91079f60211cb33033594b4",
    "ec9dc1e1aa0543c1b3fa3d2576fabffd", "6bbfe433274aef6fee5aa5543ad69055", "1f5e9cdc190854c605b533917b2c9b82",
    "858b7234b5f77661345f205b1c5ec121"
]
CONFIG = { "GAMES_PLAYED_THRESHOLD_ACCLIMATED": 15, "GAMES_PLAYED_THRESHOLD_LOW_VOLUME": 20 }

# ==============================================================================
# Part 2: Helper Functions and Data Fetching
# ==============================================================================
class TimeoutError(Exception): pass
def run_with_timeout(seconds):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            def handler(signum, frame): raise TimeoutError(f"Function '{func.__name__}' timed out")
            signal.signal(signal.SIGALRM, handler)
            signal.alarm(seconds)
            try: result = func(*args, **kwargs)
            finally: signal.alarm(0)
            return result
        return wrapper
    return decorator

def normalize_player_name(name):
    if not isinstance(name, str): return None
    name = unidecode(name.lower()); name = re.sub(r'\s+(jr|sr|ii|iii|iv)\.?$', '', name); name = re.sub(r'[^\w\s]', '', name); name = re.sub(r'\s+', ' ', name).strip()
    return name

def get_implied_prob_from_decimal(odd):
    if odd is None or pd.isna(odd) or float(odd) < 1: return None
    return 1 / float(odd)

def decimal_to_american(decimal_odds):
    if pd.isna(decimal_odds) or decimal_odds is None: return None
    if decimal_odds >= 2.00: return f"+{int((decimal_odds - 1) * 100)}"
    else: return f"-{int(100 / (decimal_odds - 1))}"

def get_current_nba_season():
    now = datetime.now()
    if now.month >= 8: start_year, end_year = now.year, now.year + 1
    else: end_year, start_year = now.year, now.year - 1
    return f"{start_year}-{str(end_year)[-2:]}"

def fetch_injury_data():
    print("--- Fetching Live Injury Data from ESPN ---")
    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()

        injury_status_dict = {}
        for team in data.get('injuries', []):
            for injury in team.get('injuries', []):
                player_name = injury.get('athlete', {}).get('displayName')
                status = injury.get('type', {}).get('description')
                if player_name and status:
                    normalized_name = normalize_player_name(player_name)
                    injury_status_dict[normalized_name] = status.title()

        print(f"Successfully fetched injury status for {len(injury_status_dict)} players.")
        return injury_status_dict
    except requests.exceptions.RequestException as e:
        print(f"Warning: Could not fetch injury data from ESPN. Error: {e}")
        return {}

def get_draft_class_ids(season_year):
    print(f"--- Fetching {season_year} Draft Class ---")
    try:
        draft_history_data = drafthistory.DraftHistory(season_year_nullable=season_year)
        df_draft = draft_history_data.get_data_frames()[0]
        return set(df_draft['PERSON_ID'].unique())
    except Exception as e:
        print(f"Could not fetch draft class for {season_year}: {e}"); return set()

def identify_team_changes(master_df):
    print("--- Identifying Players on New Teams ---")
    changed_teams_df = master_df[(master_df['TEAM_ABBREVIATION'] != master_df['CURRENT_TEAM']) & (master_df['CURRENT_TEAM'].notna()) & (master_df['TEAM_ABBREVIATION'].notna())]
    return set(changed_teams_df['PLAYER_ID'].unique())

def get_player_experience(row, rookie_ids, traded_player_ids):
    player_id, games_played_last_season, games_played_this_season = row['PLAYER_ID'], row.get('GP', 99), row.get('GP_current', 0)
    is_rookie = player_id in rookie_ids
    is_traded = player_id in traded_player_ids
    is_acclimated = games_played_this_season >= CONFIG["GAMES_PLAYED_THRESHOLD_ACCLIMATED"]
    if is_rookie: return "Traded Rookie" if is_traded and not is_acclimated else "Rookie"
    else:
        if is_traded and not is_acclimated: return "Traded Veteran"
        if games_played_last_season < CONFIG["GAMES_PLAYED_THRESHOLD_LOW_VOLUME"]: return "Low Volume"
        return "Veteran"

@run_with_timeout(MASTER_SCRAPE_TIMEOUT)
def scrape_bref_positions_by_team(season_end_year, teams_to_scrape):
    print(f"--- Scraping B-Ref rosters for {len(teams_to_scrape)} teams for season {season_end_year} ---")
    all_players_df_list = []
    chrome_options = uc.ChromeOptions(); chrome_options.add_argument('--headless'); chrome_options.add_argument('--no-sandbox'); chrome_options.add_argument('--disable-dev-shm-usage'); chrome_options.add_argument('--disable-gpu')
    driver = uc.Chrome(options=chrome_options)
    try:
        for team_abbr_bref in tqdm(teams_to_scrape, desc="Scraping B-Ref Positions"):
            for attempt in range(API_RETRY_ATTEMPTS):
                try:
                    url = f"https://www.basketball-reference.com/teams/{team_abbr_bref}/{season_end_year}.html"
                    driver.get(url)
                    WebDriverWait(driver, 30).until(EC.presence_of_element_located((By.ID, "roster")))
                    tables = pd.read_html(StringIO(driver.page_source), attrs={'id': 'roster'})
                    if tables:
                        team_df = tables[0].copy()
                        team_df['CURRENT_TEAM_BRef'] = team_abbr_bref
                        team_df.rename(columns={'Player': 'BRef_Name', 'Pos': 'DetailedPos'}, inplace=True)
                        all_players_df_list.append(team_df)
                    time.sleep(WEB_SCRAPE_DELAY)
                    break
                except Exception as e:
                    print(f"  - Attempt {attempt+1} failed for {team_abbr_bref}: {e}")
                    if attempt < API_RETRY_ATTEMPTS - 1:
                        time.sleep(WEB_SCRAPE_RETRY_DELAY)
                    else:
                        print(f"  - CRITICAL: All attempts failed for {team_abbr_bref}. Skipping team.")
    finally:
        driver.quit()
    if not all_players_df_list: print("CRITICAL ERROR: Failed to scrape any team data."); return pd.DataFrame()
    return pd.concat(all_players_df_list, ignore_index=True)

def update_and_get_live_dvp(season, data_dir, bref_positions_df, nba_positions_df):
    print("\n--- Updating and Calculating Live DvP Data ---")
    gamelogs_cache_path = os.path.join(data_dir, f"live_gamelogs_{season}.csv")
    existing_gamelogs = pd.DataFrame()
    last_date_in_cache = None
    if os.path.exists(gamelogs_cache_path):
        try:
            existing_gamelogs = pd.read_csv(gamelogs_cache_path, low_memory=False)
            existing_gamelogs['GAME_DATE'] = pd.to_datetime(existing_gamelogs['GAME_DATE'])
            if not existing_gamelogs.empty:
                last_date_in_cache = existing_gamelogs['GAME_DATE'].max()
                print(f"Loaded {len(existing_gamelogs)} cached gamelogs. Last entry is from: {last_date_in_cache.date()}")
        except Exception as e: print(f"Warning: Could not read gamelogs cache. Error: {e}")

    date_from = (last_date_in_cache + timedelta(days=1)).strftime('%m/%d/%Y') if last_date_in_cache else None

    try:
        print(f"Fetching new game logs from {date_from or 'the beginning of the season'}...")
        new_gamelogs = playergamelogs.PlayerGameLogs(season_nullable=season, date_from_nullable=date_from, timeout=API_TIMEOUT).get_data_frames()[0]
        if not new_gamelogs.empty:
            new_gamelogs['GAME_DATE'] = pd.to_datetime(new_gamelogs['GAME_DATE'])
            all_gamelogs = pd.concat([existing_gamelogs, new_gamelogs], ignore_index=True)
            all_gamelogs.drop_duplicates(subset=['GAME_ID', 'PLAYER_ID'], keep='last', inplace=True)
            all_gamelogs.to_csv(gamelogs_cache_path, index=False)
            print(f"Found {len(new_gamelogs)} new game logs. Cache updated.")
        else:
            print("No new game logs found. DvP data is up to date.")
            all_gamelogs = existing_gamelogs
    except Exception as e:
        print(f"Warning: Failed to fetch new game logs. Using cached data. Error: {e}")
        all_gamelogs = existing_gamelogs

    if all_gamelogs.empty: print("CRITICAL: No game logs available to calculate DvP."); return pd.DataFrame()

    print("Calculating season-to-date DvP stats...")

    gamelogs_with_pos = pd.DataFrame()
    if not bref_positions_df.empty:
        print("Using B-Ref for detailed positions.")
        bref_positions_df['BRef_Name_Normalized'] = bref_positions_df['BRef_Name'].apply(normalize_player_name)
        all_gamelogs['PLAYER_NAME_NORMALIZED'] = all_gamelogs['PLAYER_NAME'].apply(normalize_player_name)
        gamelogs_with_pos = pd.merge(all_gamelogs, bref_positions_df[['BRef_Name_Normalized', 'DetailedPos']], left_on='PLAYER_NAME_NORMALIZED', right_on='BRef_Name_Normalized', how='left')
        gamelogs_with_pos.rename(columns={'DetailedPos': 'Game_Position'}, inplace=True)
        gamelogs_with_pos['Game_Position'] = gamelogs_with_pos['Game_Position'].apply(lambda x: x.split('-')[0] if isinstance(x, str) and x else 'Unknown')
    elif not nba_positions_df.empty:
        print("B-Ref scrape failed. Using NBA_API for general positions (G, F, C).")
        gamelogs_with_pos = pd.merge(all_gamelogs, nba_positions_df, on='PLAYER_ID', how='left')
        gamelogs_with_pos.rename(columns={'POSITION': 'Game_Position'}, inplace=True)
        gamelogs_with_pos['Game_Position'] = gamelogs_with_pos['Game_Position'].apply(lambda x: x.split('-')[0] if isinstance(x, str) and x else 'Unknown')
    else:
        print("CRITICAL WARNING: No position data available. DvP will be inaccurate.")
        # Create a dummy column so the rest of the script doesn't break
        all_gamelogs['Game_Position'] = 'Unknown'
        gamelogs_with_pos = all_gamelogs

    opponent_df = gamelogs_with_pos[['GAME_ID', 'TEAM_ABBREVIATION']].drop_duplicates()
    gamelogs_with_pos = pd.merge(gamelogs_with_pos, opponent_df.rename(columns={'TEAM_ABBREVIATION': 'OPPONENT_ABBR'}), on='GAME_ID')
    gamelogs_with_pos = gamelogs_with_pos[gamelogs_with_pos['TEAM_ABBREVIATION'] != gamelogs_with_pos['OPPONENT_ABBR']]
    for stat in ['PTS', 'REB', 'AST']: gamelogs_with_pos[stat] = pd.to_numeric(gamelogs_with_pos[stat], errors='coerce')
    dvp = gamelogs_with_pos.groupby(['OPPONENT_ABBR', 'Game_Position'])[['PTS', 'REB', 'AST']].mean().reset_index()
    dvp_pivot = dvp.pivot_table(index='OPPONENT_ABBR', columns='Game_Position', values=['PTS', 'REB', 'AST'])
    dvp_pivot.columns = [f'OPP_DvP_{stat}_vs_{pos}' for stat, pos in dvp_pivot.columns]
    print("--- DvP Calculation Complete ---")
    return dvp_pivot

feature_cache = {}
def get_live_features_for_player(player_id, stats_season):
    if player_id in feature_cache: return feature_cache[player_id]
    print(f"  - Engineering features for player ID: {player_id}...")
    try:
        logs = playergamelog.PlayerGameLog(player_id=player_id, season=stats_season, timeout=API_TIMEOUT).get_data_frames()[0]
        if logs.empty or len(logs) < 3: return None
        logs['GAME_DATE'] = pd.to_datetime(logs['GAME_DATE']); logs = logs.sort_values('GAME_DATE', ascending=True)
        features = {}
        stats_to_roll = ['PTS', 'REB', 'AST', 'MIN', 'FGA', 'FTA', 'FG3A']
        windows = [3, 5, 10]
        for stat in stats_to_roll:
            for w in windows: features[f'ROLL_AVG_{stat}_{w}'] = logs[stat].tail(w).mean()

        # --- CORRECTED: Backfill missing longer-term averages with shorter-term ones in the correct order ---
        for stat in stats_to_roll:
            # First, ensure 5-game avg is filled if it's missing
            if pd.isna(features.get(f'ROLL_AVG_{stat}_5')):
                features[f'ROLL_AVG_{stat}_5'] = features.get(f'ROLL_AVG_{stat}_3')
            # Now, ensure 10-game avg is filled with the (potentially newly filled) 5-game avg
            if pd.isna(features.get(f'ROLL_AVG_{stat}_10')):
                features[f'ROLL_AVG_{stat}_10'] = features.get(f'ROLL_AVG_{stat}_5')

        features['DAYS_OF_REST'] = min((datetime.now(timezone.utc).replace(tzinfo=None) - logs.iloc[-1]['GAME_DATE']).days, 7)
        feature_cache[player_id] = features
        return features
    except Exception as e:
        print(f"    - Could not fetch game logs for player {player_id}. Error: {e}"); return None

# ==============================================================================
# Part 3: The Main Prediction Engine
# ==============================================================================
def run_daily_projections():
    try:
        drive.mount('/content/drive', force_remount=True)
        DATA_DIR = "/content/drive/MyDrive/NBA/Historical Data/"
        MODEL_DIR = "/content/drive/MyDrive/NBA/Training Model"
        OUTPUT_DIR = "/content/drive/MyDrive/NBA/Predictions"
        os.makedirs(MODEL_DIR, exist_ok=True); os.makedirs(OUTPUT_DIR, exist_ok=True); os.makedirs(DATA_DIR, exist_ok=True)
    except Exception as e: print(f"Could not mount Google Drive. Aborting. Error: {e}"); return

    injury_data = fetch_injury_data()
    print("\n--- Loading Trained Models & Performance Data ---")
    regression_props = ['PTS', 'REB', 'AST', 'P+R', 'P+A', 'R+A', 'PRA']; classification_props = ['DD', 'TD']
    models = {}; mae_values = {}
    try:
        for prop in regression_props: models[prop] = joblib.load(os.path.join(MODEL_DIR, f"nba_{prop.lower()}_ridge_model.pkl"))
        for prop in classification_props: models[prop] = joblib.load(os.path.join(MODEL_DIR, f"nba_{prop.lower()}_logreg_model.pkl"))
        with open(os.path.join(MODEL_DIR, "mae_values.json"), 'r') as f: mae_values = json.load(f)
    except FileNotFoundError as e: print(f"CRITICAL ERROR: Could not load model files. Please run Script 2. Missing: {e}"); return

    CURRENT_SEASON = get_current_nba_season()
    eastern = pytz.timezone('US/Eastern'); now_et = datetime.now(eastern)

    print("\n--- Step 1: Fetching today's game schedule from Odds API ---")
    start_of_day_et = now_et.replace(hour=8, minute=0, second=0, microsecond=0); end_of_day_et = start_of_day_et + timedelta(days=1)
    start_utc_str = start_of_day_et.astimezone(pytz.utc).strftime('%Y-%m-%dT%H:%M:%SZ'); end_utc_str = end_of_day_et.astimezone(pytz.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    working_key, events = None, []
    for i, api_key in enumerate(ODDS_API_KEYS):
        try:
            response = requests.get('https://api.the-odds-api.com/v4/sports/basketball_nba/events', params={'apiKey': api_key, 'commenceTimeFrom': start_utc_str, 'commenceTimeTo': end_utc_str}, timeout=30)
            response.raise_for_status(); events = response.json(); working_key = api_key; break
        except Exception as e: print(f"--- API Key #{i+1} failed. Error: {e} ---")
    if not working_key: print("--- CRITICAL: All API keys failed. ---"); return
    if not events: print("No NBA games scheduled for today."); return
    team_name_map = {team['full_name']: team['abbreviation'] for team in teams.get_teams()}
    games_today = [{'home_team_abbr': team_name_map.get(e['home_team']), 'away_team_abbr': team_name_map.get(e['away_team'])} for e in events if team_name_map.get(e['home_team']) and team_name_map.get(e['away_team'])]
    games_today = [dict(t) for t in {tuple(d.items()) for d in games_today}]

    print("\n--- Step 2: Acquiring Player Rosters & Positions (with self-healing) ---")
    today_str = now_et.strftime('%Y-%m-%d')
    current_year_start = int(CURRENT_SEASON.split('-')[0])
    POSITIONS_SEASON_END_YEAR = current_year_start + 1
    teams_playing_today = {g['home_team_abbr'] for g in games_today} | {g['away_team_abbr'] for g in games_today}
    bref_team_map = {"BKN": "BRK", "PHX": "PHO", "CHA": "CHO"}; teams_to_scrape_bref = [bref_team_map.get(team, team) for team in teams_playing_today]

    bref_df = pd.DataFrame()
    if USE_BASKETBALL_REFERENCE:
        for attempt in range(MASTER_SCRAPE_ATTEMPTS):
            try:
                print(f"Master scraping attempt {attempt + 1}/{MASTER_SCRAPE_ATTEMPTS}...")
                bref_df = scrape_bref_positions_by_team(season_end_year=POSITIONS_SEASON_END_YEAR, teams_to_scrape=teams_to_scrape_bref)
                if not bref_df.empty and set(bref_df['CURRENT_TEAM_BRef'].unique()) == set(teams_to_scrape_bref):
                    print("✅ Successfully scraped all required team rosters.")
                    break
                else:
                    print(f"  - Warning: Scrape attempt {attempt+1} was incomplete. Retrying...")
            except TimeoutError as e:
                print(f"  - CRITICAL: Scrape attempt {attempt+1} timed out. Error: {e}")
            except Exception as e:
                print(f"  - CRITICAL: Scrape attempt {attempt+1} failed with an unexpected error: {e}")
            if attempt < MASTER_SCRAPE_ATTEMPTS - 1:
                print("    -> Waiting before next master attempt...")
                time.sleep(WEB_SCRAPE_RETRY_DELAY)

    if bref_df.empty:
        if USE_BASKETBALL_REFERENCE:
            print("CRITICAL FAILURE: Could not scrape rosters after multiple master attempts. Continuing with NBA_API positions.")
        else:
            print("Skipping Basketball-Reference scrape as configured. Continuing with NBA_API positions.")

    use_previous_season_stats = today_str < f"{current_year_start}-12-01"
    STATS_SEASON_STRING = f"{current_year_start - 1}-{str(current_year_start)[-2:]}" if use_previous_season_stats else CURRENT_SEASON
    nba_df = leaguedashplayerstats.LeagueDashPlayerStats(season=STATS_SEASON_STRING, per_mode_detailed="PerGame").get_data_frames()[0]
    nba_df_current = leaguedashplayerstats.LeagueDashPlayerStats(season=CURRENT_SEASON, per_mode_detailed="PerGame").get_data_frames()[0] if use_previous_season_stats and CURRENT_SEASON != STATS_SEASON_STRING else nba_df

    print("--- Fetching NBA Player Positions for Fallback ---")
    try:
        all_players_list = players.get_players()
        nba_positions_df = pd.DataFrame(all_players_list)
        nba_positions_df.rename(columns={'id': 'PLAYER_ID', 'full_name': 'PLAYER_NAME'}, inplace=True)
        # This static list does not have positions, so we will handle that later.
    except Exception as e:
        print(f"CRITICAL: Could not load static player list. Error: {e}")
        nba_positions_df = pd.DataFrame()


    dvp_df = update_and_get_live_dvp(CURRENT_SEASON, DATA_DIR, bref_df, nba_positions_df)

    print("\n--- Step 3: Fetching player prop odds from Odds API ---")
    MARKETS_TO_KEEP = {'player_points': 'PTS', 'player_rebounds': 'REB', 'player_assists': 'AST', 'player_points_rebounds_assists': 'PRA', 'player_points_rebounds': 'P+R', 'player_points_assists': 'P+A', 'player_rebounds_assists': 'R+A', 'player_double_double': 'DD', 'player_triple_double': 'TD'}
    all_props = [];
    key_usage = {}
    print("Checking API key usage...")
    for key in ODDS_API_KEYS:
        try:
            usage_response = requests.get('https://api.the-odds-api.com/v4/sports', params={'apiKey': key})
            remaining = int(usage_response.headers.get('x-requests-remaining', 0))
            key_usage[key] = remaining
            print(f"  - Key ending in ...{key[-4:]} has {remaining} requests remaining.")
        except Exception as e:
            key_usage[key] = 0
            print(f"  - Could not check usage for key ending in ...{key[-4:]}. Error: {e}")

    current_key_index = 0
    for event in events:
        event_id = event['id']
        odds_fetched = False
        while not odds_fetched and current_key_index < len(ODDS_API_KEYS):
            api_key = ODDS_API_KEYS[current_key_index]
            if key_usage.get(api_key, 0) < len(events):
                print(f"Skipping key ...{api_key[-4:]}, not enough requests remaining.")
                current_key_index += 1
                continue
            params = {'apiKey': api_key, 'regions': 'us', 'markets': ','.join(MARKETS_TO_KEEP.keys()), 'bookmakers': 'draftkings,fanduel'}
            try:
                response = requests.get(f'https://api.the-odds-api.com/v4/sports/basketball_nba/events/{event_id}/odds', params=params, timeout=30)
                response.raise_for_status()
                for bookmaker in response.json().get('bookmakers', []):
                    for market in bookmaker.get('markets', []):
                        prop_type = MARKETS_TO_KEEP.get(market['key'])
                        if prop_type:
                            for outcome in market.get('outcomes', []):
                                all_props.append({ 'Game': f"{team_name_map.get(event['away_team'])} @ {team_name_map.get(event['home_team'])}", 'Game_Time': event['commence_time'], 'Player': outcome['description'], 'Prop': prop_type, 'Line': outcome.get('point'), 'Over_Odds': outcome['price'] if outcome['name'] == 'Over' else None, 'Under_Odds': outcome['price'] if outcome['name'] == 'Under' else None, 'Yes_Odds': outcome['price'] if outcome['name'] == 'Yes' else None, 'No_Odds': outcome['price'] if outcome['name'] == 'No' else None, 'Bookmaker': bookmaker['title'] })
                odds_fetched = True
            except requests.exceptions.RequestException as e:
                print(f"Error fetching odds for event {event_id}. Key #{current_key_index + 1} failed. Error: {e}")
                current_key_index += 1
        if not odds_fetched and current_key_index >= len(ODDS_API_KEYS):
            print("All Odds API keys failed. Aborting odds fetch for remaining games.")
            break

    if not all_props: print("No player props were available. Aborting."); return
    df = pd.DataFrame(all_props); df['match_key'] = df['Player'].apply(normalize_player_name)
    odds_df_ou = df[df['Prop'].isin(regression_props)].groupby(['match_key', 'Player', 'Prop', 'Line', 'Bookmaker', 'Game', 'Game_Time']).agg({'Over_Odds': 'first', 'Under_Odds': 'first'}).reset_index()
    odds_df_yn = df[df['Prop'].isin(classification_props)].groupby(['match_key', 'Player', 'Prop', 'Bookmaker', 'Game', 'Game_Time']).agg({'Yes_Odds': 'first', 'No_Odds': 'first'}).reset_index()
    odds_df_yn['Implied_Prob_Yes'] = odds_df_yn['Yes_Odds'].apply(get_implied_prob_from_decimal)
    odds_df = pd.concat([odds_df_ou, odds_df_yn], ignore_index=True)
    for col in ['Over_Odds', 'Under_Odds', 'Yes_Odds', 'No_Odds']: odds_df[col] = odds_df[col].apply(decimal_to_american)
    odds_df['Game_Time'] = pd.to_datetime(odds_df['Game_Time']).dt.tz_convert('US/Eastern').dt.strftime('%-I:%M %p EST')

    print("\n--- Step 4: Preparing Data for Projections ---")
    nba_df['merge_key'] = nba_df['PLAYER_NAME'].apply(normalize_player_name)

    if not bref_df.empty:
        bref_df['merge_key'] = bref_df['BRef_Name'].apply(normalize_player_name)
        master_df = pd.merge(nba_df, bref_df, on='merge_key', how='inner')
        reverse_bref_map = {v: k for k, v in bref_team_map.items()}
        master_df['CURRENT_TEAM'] = master_df['CURRENT_TEAM_BRef'].apply(lambda x: reverse_bref_map.get(x, x))
    else:
        master_df = nba_df.copy()
        master_df['DetailedPos'] = 'Unknown'
        master_df['CURRENT_TEAM'] = master_df['TEAM_ABBREVIATION']


    if use_previous_season_stats: master_df['GP_current'] = master_df['PLAYER_ID'].map(nba_df_current.set_index('PLAYER_ID')['GP']).fillna(0)
    else: master_df['GP_current'] = master_df['GP']
    rookie_ids = get_draft_class_ids(str(current_year_start))
    traded_player_ids = identify_team_changes(master_df)

    players_with_odds = odds_df['match_key'].unique()
    players_to_project_df = master_df[master_df['merge_key'].isin(players_with_odds)]
    print(f"\n--- Step 5: Generating Projections for {len(players_to_project_df)} Players with Active Props ---")

    all_projections = []
    for _, player in players_to_project_df.iterrows():
        time.sleep(INTER_PLAYER_DELAY)
        player_features = get_live_features_for_player(player['PLAYER_ID'], CURRENT_SEASON)
        if not player_features: continue

        opp_team_abbr = next((g['home_team_abbr'] if player['CURRENT_TEAM'] == g['away_team_abbr'] else g['away_team_abbr'] for g in games_today if player['CURRENT_TEAM'] in [g['home_team_abbr'], g['away_team_abbr']]), None)
        if not opp_team_abbr: continue

        player_pos = player.get('DetailedPos', 'Unknown')
        if not player_pos or not isinstance(player_pos, str):
            player_pos = 'Unknown'
        else:
            player_pos = player_pos.split('-')[0]

        for stat in ['PTS', 'REB', 'AST']:
            dvp_feature_name = f'OPP_DvP_{stat}_vs_POS'
            dvp_lookup_col = f'OPP_DvP_{stat}_vs_{player_pos}'
            if not dvp_df.empty and opp_team_abbr in dvp_df.index and dvp_lookup_col in dvp_df.columns:
                player_features[dvp_feature_name] = dvp_df.loc[opp_team_abbr, dvp_lookup_col]
            else:
                player_features[dvp_feature_name] = np.nan

        projections_for_player = {'Player': player['PLAYER_NAME'], 'Team': player['CURRENT_TEAM'], 'Pos': player['DetailedPos'], 'match_key': player['merge_key'], 'Experience': get_player_experience(player, rookie_ids, traded_player_ids)}
        projections_for_player['Injury'] = injury_data.get(player['merge_key'], 'Healthy')

        TRAINING_FEATURES = [
            'ROLL_AVG_PTS_3', 'ROLL_AVG_REB_3', 'ROLL_AVG_AST_3', 'ROLL_AVG_PTS_5',
            'ROLL_AVG_REB_5', 'ROLL_AVG_AST_5', 'ROLL_AVG_PTS_10', 'ROLL_AVG_REB_10',
            'ROLL_AVG_AST_10', 'ROLL_AVG_MIN_10', 'ROLL_AVG_FGA_10', 'ROLL_AVG_FTA_10',
            'ROLL_AVG_FG3A_10', 'DAYS_OF_REST',
            'OPP_DvP_PTS_vs_POS', 'OPP_DvP_REB_vs_POS', 'OPP_DvP_AST_vs_POS'
        ]
        feature_df = pd.DataFrame([player_features])
        feature_df = feature_df[TRAINING_FEATURES]

        for prop in regression_props: projections_for_player[f'Proj_{prop}'] = round(max(0, models[prop].predict(feature_df)[0]), 1)
        for prop in classification_props: projections_for_player[f'Proj_{prop}_%'] = f"{models[prop].predict_proba(feature_df)[0][1]:.1%}"
        all_projections.append(projections_for_player)

    if not all_projections: print("\n--- WARNING: No projections were generated. ---"); return
    projections_df = pd.DataFrame(all_projections)

    print("\n--- Step 6: Calculating Edge and Generating Output Files ---")
    melted_projections = projections_df.melt(id_vars=['match_key', 'Player', 'Team', 'Injury', 'Experience', 'Pos'], value_vars=[c for c in projections_df.columns if c.startswith('Proj')], var_name='Prop_Name', value_name='Projection')
    melted_projections['Prop'] = melted_projections['Prop_Name'].str.replace('Proj_', '').str.replace('_%', '')
    master_long_df = pd.merge(melted_projections, odds_df, on=['match_key', 'Prop'], how='left', suffixes=('_proj', '_odds'))
    master_long_df.dropna(subset=['Game'], inplace=True)
    master_long_df['Player'] = master_long_df['Player_proj'].fillna(master_long_df['Player_odds']); master_long_df.drop(columns=['Player_proj', 'Player_odds'], inplace=True, errors='ignore')

    # --- NEW: Robust Sanity Check to filter out scrambled player data ---
    def get_teams_from_game_str(game_str):
        if not isinstance(game_str, str) or '@' not in game_str: return [], []
        away, home = game_str.split(' @ ')
        return away.strip(), home.strip()

    teams_in_game = master_long_df['Game'].apply(get_teams_from_game_str)
    master_long_df['AwayTeam'] = teams_in_game.apply(lambda x: x[0])
    master_long_df['HomeTeam'] = teams_in_game.apply(lambda x: x[1])

    initial_rows = len(master_long_df)
    master_long_df = master_long_df[master_long_df.apply(lambda row: row['Team'] in [row['AwayTeam'], row['HomeTeam']], axis=1)]
    filtered_rows = initial_rows - len(master_long_df)
    if filtered_rows > 0:
        print(f"  - Sanity Check: Removed {filtered_rows} rows with incorrect player/team assignments due to placeholder API data.")
    master_long_df.drop(columns=['AwayTeam', 'HomeTeam'], inplace=True)

    master_long_df['Projection_Num'] = pd.to_numeric(master_long_df['Projection'].astype(str).str.replace('%', ''), errors='coerce')
    master_long_df.drop(columns=['Projection'], inplace=True, errors='ignore')

    yn_mask = master_long_df['Prop'].isin(classification_props)
    ou_mask = master_long_df['Prop'].isin(regression_props)

    master_long_df['DD & TD\nModel Score'] = np.nan
    master_long_df.loc[yn_mask, 'DD & TD\nModel Score'] = master_long_df.loc[yn_mask, 'Projection_Num']
    master_long_df.loc[yn_mask, 'Projection_Num'] = np.nan

    for prop, mae in mae_values.items():
        mask = master_long_df['Prop'] == prop
        master_long_df.loc[mask, 'Proj Range (Low)'] = (master_long_df.loc[mask, 'Projection_Num'] - mae).round(1).apply(lambda x: max(0, x))
        master_long_df.loc[mask, 'Proj Range (High)'] = (master_long_df.loc[mask, 'Projection_Num'] + mae).round(1)
    master_long_df['Proj Range'] = master_long_df.apply(lambda row: f"{row['Proj Range (Low)']}-{row['Proj Range (High)']}" if pd.notna(row['Proj Range (Low)']) else "", axis=1)

    master_long_df['Edge'] = None; master_long_df['Bet'] = 'Pass'
    master_long_df.loc[ou_mask, 'Edge'] = ((master_long_df.loc[ou_mask, 'Projection_Num'] - master_long_df.loc[ou_mask, 'Line']) / master_long_df.loc[ou_mask, 'Line']) * 100
    master_long_df.loc[ou_mask, 'Bet'] = master_long_df.loc[ou_mask, 'Edge'].apply(lambda x: 'Over' if pd.notna(x) and x > 10 else ('Under' if pd.notna(x) and x < -10 else 'Pass'))
    if 'Implied_Prob_Yes' in master_long_df.columns:
        master_long_df.loc[yn_mask, 'Edge'] = master_long_df.loc[yn_mask, 'DD & TD\nModel Score'] - (master_long_df.loc[yn_mask, 'Implied_Prob_Yes'].fillna(0) * 100)
    else:
        master_long_df.loc[yn_mask, 'Edge'] = None
    master_long_df.loc[yn_mask, 'Bet'] = master_long_df.loc[yn_mask, 'Edge'].apply(lambda x: 'Yes' if pd.notna(x) and x > 5 else ('No' if pd.notna(x) and x < -5 else 'Pass'))
    if 'Implied_Prob_Yes' in master_long_df.columns: master_long_df.drop(columns=['Implied_Prob_Yes'], inplace=True)


    master_long_df['Alt Line'] = 'Pass'
    alt_line_mask = (master_long_df['Bet'] == 'Over') & (master_long_df['Proj Range (Low)'].notna())
    alt_line_values = np.floor(pd.to_numeric(master_long_df.loc[alt_line_mask, 'Proj Range (Low)'], errors='coerce'))
    valid_alt_line_mask = alt_line_values < master_long_df.loc[alt_line_mask, 'Line']
    master_long_df.loc[alt_line_mask & valid_alt_line_mask, 'Alt Line'] = 'Over ' + alt_line_values[valid_alt_line_mask].astype(int).astype(str)

    master_long_df['DD/TD Odds'] = master_long_df['Yes_Odds']
    master_long_df.rename(columns={'Projection_Num': 'Projection', 'Over_Odds': 'Over Odds', 'Under_Odds': 'Under Odds'}, inplace=True)
    master_long_df.drop(columns=['Proj Range (Low)', 'Proj Range (High)', 'Yes_Odds', 'No_Odds'], inplace=True, errors='ignore')

    cols_to_round = ['DD & TD\nModel Score', 'Line', 'Projection', 'Edge']
    for col in cols_to_round:
        if col in master_long_df.columns:
            master_long_df[col] = pd.to_numeric(master_long_df[col], errors='coerce').round(1)

    game_date_str = pd.to_datetime(events[0]['commence_time']).tz_convert('US/Eastern').strftime('%m-%d-%Y')
    output_filename = f"{game_date_str} NBA Props Predictions.xlsx"
    full_output_path = os.path.join(OUTPUT_DIR, output_filename)
    with pd.ExcelWriter(full_output_path, engine='xlsxwriter') as writer:
        workbook  = writer.book
        header_format = workbook.add_format({'bold': True, 'align': 'center', 'valign': 'vcenter', 'fg_color': '#D7E4BC', 'border': 1, 'text_wrap': True})
        center_format = workbook.add_format({'align': 'center', 'valign': 'vcenter', 'border': 1})
        injured_format = workbook.add_format({'bg_color': '#FFEB9C', 'align': 'center', 'valign': 'vcenter', 'border': 1}) # Yellow fill for injured players

        ev_cols = ['Game_Time', 'Game', 'Player', 'Team', 'Experience', 'Prop', 'DD & TD\nModel Score', 'Line', 'Projection', 'Proj Range', 'Alt Line', 'Edge', 'Bet', 'Over Odds', 'Under Odds', 'DD/TD Odds', 'Injury']
        final_cols = ['Player', 'Team', 'Experience', 'Prop', 'DD & TD\nModel Score', 'Line', 'Projection', 'Proj Range', 'Alt Line', 'Edge', 'Bet', 'Over Odds', 'Under Odds', 'DD/TD Odds', 'Bookmaker', 'Injury']

        all_ev_bets_df = master_long_df[master_long_df['Bet'] != 'Pass'].copy()
        if not all_ev_bets_df.empty:
            for bookmaker in ['DraftKings', 'FanDuel']:
                book_df = all_ev_bets_df[all_ev_bets_df['Bookmaker'] == bookmaker]
                if book_df.empty: continue
                sorted_games = sorted(games_today, key=lambda g: (g['home_team_abbr'], g['away_team_abbr']))
                game_dfs = [game_df.sort_values(by='Edge', key=abs, ascending=False) for game_tuple in sorted_games if not (game_df := book_df[book_df['Game'] == f"{game_tuple['away_team_abbr']} @ {game_tuple['home_team_abbr']}"]).empty]
                if game_dfs:
                    final_ev_df = pd.concat(game_dfs, ignore_index=True)
                    final_ev_df = final_ev_df.reindex(columns=ev_cols)
                    sheet_name = f"{bookmaker}_EV_Bets"
                    final_ev_df.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=1)
                    worksheet = writer.sheets[sheet_name]

                    for col_num, value in enumerate(final_ev_df.columns.values):
                        worksheet.write(0, col_num, value, header_format)
                        data_len = final_ev_df[value].astype(str).map(len).max() if not final_ev_df[value].isnull().all() else 0
                        header_len = max([len(s) for s in value.split('\n')]) if '\n' in value else len(value)
                        column_width = max(data_len, header_len, 10) + 2
                        worksheet.set_column(col_num, col_num, column_width, center_format)
                    worksheet.set_row(0, 30)

                    # Add conditional formatting for injured players
                    injury_col_letter = xl_col_to_name(ev_cols.index('Injury'))
                    worksheet.conditional_format(f'A2:{xl_col_to_name(len(ev_cols)-1)}{len(final_ev_df)+1}', {
                        'type': 'formula',
                        'formula': f'${injury_col_letter}2 <> "Healthy"',
                        'format': injured_format
                    })


        for game in sorted(games_today, key=lambda g: (g['home_team_abbr'], g['away_team_abbr'])):
            home_abbr, away_abbr = game['home_team_abbr'], game['away_team_abbr']
            sheet_name = f"{away_abbr}_vs_{home_abbr}_Full_Data"
            game_df = master_long_df[master_long_df['Game'] == f"{away_abbr} @ {home_abbr}"].copy()
            if game_df.empty: continue
            game_long_df = game_df.reindex(columns=final_cols).sort_values(by=['Player', 'Prop'])
            game_long_df.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=1)
            worksheet = writer.sheets[sheet_name]

            for col_num, value in enumerate(game_long_df.columns.values):
                worksheet.write(0, col_num, value, header_format)
                data_len = game_long_df[value].astype(str).map(len).max() if not game_long_df[value].isnull().all() else 0
                header_len = max([len(s) for s in value.split('\n')]) if '\n' in value else len(value)
                column_width = max(data_len, header_len, 10) + 2
                worksheet.set_column(col_num, col_num, column_width, center_format)
            worksheet.set_row(0, 30)

            # Add conditional formatting for injured players
            injury_col_letter = xl_col_to_name(final_cols.index('Injury'))
            worksheet.conditional_format(f'A2:{xl_col_to_name(len(final_cols)-1)}{len(game_long_df)+1}', {
                'type': 'formula',
                'formula': f'${injury_col_letter}2 <> "Healthy"',
                'format': injured_format
            })


    print(f"\n✅ Successfully exported all data to: {full_output_path}")

if __name__ == "__main__":
    pd.set_option('display.max_rows', 200); pd.set_option('display.max_columns', None); pd.set_option('display.width', 1000)
    run_daily_projections()

Mounted at /content/drive
--- Fetching Live Injury Data from ESPN ---
Successfully fetched injury status for 94 players.

--- Loading Trained Models & Performance Data ---

--- Step 1: Fetching today's game schedule from Odds API ---

--- Step 2: Acquiring Player Rosters & Positions (with self-healing) ---
Skipping Basketball-Reference scrape as configured. Continuing with NBA_API positions.
--- Fetching NBA Player Positions for Fallback ---

--- Updating and Calculating Live DvP Data ---
Loaded 12893 cached gamelogs. Last entry is from: 2025-11-06
Fetching new game logs from 11/07/2025...
No new game logs found. DvP data is up to date.
Calculating season-to-date DvP stats...
B-Ref scrape failed. Using NBA_API for general positions (G, F, C).


KeyError: 'Game_Position'